# Desarrollo e implementacion de tesis

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import evaluate

/home/juan/anaconda3/envs/tesis_gpt2/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
# Especificamos el modelo que queremos: "gpt2"
model_name = "gpt2"

# Cargar el tokenizador
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Cargar el modelo
# AutoModelForCausalLM es para generación de texto, que es lo que hace GPT-2
model = AutoModelForCausalLM.from_pretrained(model_name)

# (Opcional pero recomendado) Mover el modelo a la GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Modelo {model_name} cargado en {device}.")

Modelo gpt2 cargado en cuda.


# Comienzo del experimento :D

# Caso de prueba modelo sin intervencion

### Metrica R1: Caída de Precisión en Clasificación

# Modelo GPT 2 SMALL

## 0) imports y configuraciones iniciales globales 

Este primer bloque prepara el entorno para correr los experimentos de R1 (MCQ) con modelos GPT-2 usando TransformerLens. No “hace” evaluación aún; deja listas las dependencias, parámetros y semillas para que las celdas siguientes puedan cargar datos/modelo y evaluar de forma estable y reproducible.

### Imports

* os, re, random, difflib: utilidades del sistema, regex y comparaciones de texto.
* tqdm: barras de progreso.
* HookedTransformer (TransformerLens): envoltorio del modelo para hooks y análisis.


Nota: Se establece TRANSFORMERS_NO_TORCHVISION=1 para evitar cargar torchvision (no requerido).

### Configuración principal

* DEVICE: selección automática de cuda (si disponible) o cpu.

* SEED: semilla base para reproducibilidad.

### Parámetros de R1 (MCQ)

* NUM_DISTRACTORS = 3: número de distractores por ítem (1 correcta + 3 distractores), es decir en cada pregunta de opción múltiple (MCQ) hay 1 respuesta correcta y N respuestas incorrectas (distractores).

* BAD_SUBSTRINGS: lista de subcadenas “ruidosas” para filtrar targets/hypernyms demasiado genéricos.

In [4]:
import os, re, random, difflib
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm

# Evita que Transformers intente importar torchvision (no lo necesitamos)
os.environ.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")

from transformer_lens import HookedTransformer

# ---- Config principal ----
R1_PATH = "Data/dataset_R1_en.csv"         # <-- Ajusta la ruta si está en otra carpeta
MODEL_NAME = "gpt2-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

# MCQ: 1 correcta + N distractores
NUM_DISTRACTORS = 3

# Filtros opcionales de targets/hypernyms demasiado genéricos o ruidosos
BAD_SUBSTRINGS = {
    "thing", "object", "stuff", "device type", "capability", "position",
    "most often", "mostly", "kind of", "nickname", "person", "being",
    "animal", "comics character", "fictional", "concept", "common liquid",
    "non fluidlike", "set of", "linear unit", "platform", "place", "area",
    "color", "tone", "sound", "pitch", "material", "organic matter"
}

# Seed reproducible
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 1) Utilidades communes Base y Hooks

Esta sección define funciones de preprocesamiento y filtrado que usamos antes de evaluar el modelo. Todas operan sobre cadenas y ayudan a construir prompts limpios y opciones de respuesta (distractores) razonables.

* ensure_space_prefix: Garantiza que la continuación (el target_token) empiece con espacio. Porque GPT-2 usa BPE con tokens que suelen incluir el espacio precedente. Sin ese espacio, la probabilidad del token cambia y sesga la evaluación.
* normalize: Normaliza texto para comparaciones: minúsculas + trim + colapsa espacios. Para evita falsos negativos al comparar strings con diferencias irrelevantes.
* looks_bad: Devuelve True si el string contiene alguna subcadena “ruidosa” definida en BAD_SUBSTRINGS
* too_similar: Usa difflib.SequenceMatcher para medir similitud de texto y marca True si la similitud ≥ thr. Al muestrear distractores, evitamos opciones casi idénticas al oro (que harían la pregunta trivial o ambigua).
* extract_concept_from_prompt: intenta extraer el concepto X del prompt cuando el CSV no trae la columna concept

In [5]:

def ensure_space_prefix(tok: str) -> str:
    """
    Asegura que la continuación empiece con espacio. GPT-2 suele tokenizar con espacio previo.
    """
    tok = str(tok).rstrip()
    return tok if tok.startswith(" ") else (" " + tok)

def normalize(s: str) -> str:
    return " ".join(str(s).lower().strip().split())

def looks_bad(s: str) -> bool:
    s_norm = normalize(s)
    return any(bad in s_norm for bad in BAD_SUBSTRINGS)

def too_similar(a: str, b: str, thr: float = 0.8) -> bool:
    return difflib.SequenceMatcher(None, a, b).ratio() >= thr

def extract_concept_from_prompt(prompt: str) -> str | None:
    """
    Heurística de respaldo por si no viene la columna 'concept'.
    Busca patrones típicos como: "A X is a", "An X is a", "The X is a".
    """
    p = prompt.strip()

    # 1) "A tool is a", "An apple is a", "The hammer is a"
    m = re.match(r"^(?:A|An|The)\s+([A-Za-z_]+)\b.*?\bis\s+(?:a|an)\b", p, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()

    # 2) "X is a" (fallback)
    m = re.match(r"^([A-Za-z_]+)\b.*?\bis\s+(?:a|an)\b", p, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()

    return None

## 2) CARGA DE MODELO Y LECTURA/LIMPIEZA R1 

* HookedTransformer lee y descarga el modelo que desde la libria de tranformer lents para asi poder intervenir
* ensure_space_prefix como se explico antes normaliza cada parte de los tokens de target
* filtramos los tokens conrespuestas genericas
* extraemos el concepto del dataframe para generar el test al modelo
*  ALL_TARGETS: Construye un conjunto único de posibles respuestas (targets) para escoger distractores. Y despues, sample_distractors elige N distractores que no sean iguales ni demasiado parecidos al correcto; si faltan candidatos, usa un fallback menos estricto.

In [6]:
print(f"[INFO] Cargando modelo '{MODEL_NAME}' en {DEVICE} ...")
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
print("[INFO] Modelo cargado.")

print(f"[INFO] Cargando dataset R1 desde: {R1_PATH}")
df = pd.read_csv(R1_PATH)

# Chequeo columnas mínimas
needed_cols = {"prompt_cloze", "target_token"}
missing = needed_cols - set(df.columns)
assert not missing, f"Faltan columnas necesarias en el CSV: {missing}"

# Normaliza target con espacio inicial
df["target_token"] = df["target_token"].astype(str).apply(ensure_space_prefix)

# (Opcional) Filtrado por BAD_SUBSTRINGS
orig_n = len(df)
if "hypernym" in df.columns:
    df = df[~df["hypernym"].astype(str).apply(looks_bad)].copy()
df = df[~df["target_token"].astype(str).apply(looks_bad)].copy()
df = df.drop_duplicates(subset=["prompt_cloze", "target_token"]).reset_index(drop=True)
print(f"[INFO] Filtrado opcional: {orig_n} -> {len(df)} filas.")

# Añadir columna 'concept' si no existe
if "concept" not in df.columns:
    df["concept"] = df["prompt_cloze"].apply(extract_concept_from_prompt)

# Limpieza de concept nulo (si falla la heurística, no lo uses para CAV, pero se puede evaluar baseline igual)
n_concept_none = df["concept"].isna().sum()
if n_concept_none > 0:
    print(f"[WARN] {n_concept_none} filas sin 'concept' inferible; baseline ok, CAV usará las que sí tienen.")

# Construye pool de targets para MCQ
ALL_TARGETS = sorted(set(df["target_token"].tolist()), key=lambda x: x)

def sample_distractors(gold: str, k: int = NUM_DISTRACTORS):
    gold_norm = normalize(gold)
    cands = [t for t in ALL_TARGETS if normalize(t) != gold_norm and not too_similar(normalize(t), gold_norm)]
    if len(cands) < k:  # fallback
        cands = [t for t in ALL_TARGETS if normalize(t) != gold_norm]
    k = min(k, len(cands))
    return random.sample(cands, k) if k > 0 else []

[INFO] Cargando modelo 'gpt2-small' en cuda ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
[INFO] Modelo cargado.
[INFO] Cargando dataset R1 desde: Data/dataset_R1_en.csv
[INFO] Filtrado opcional: 348 -> 322 filas.


## 3) BASELINE: LOG-PROB DE LA CONTINUACIÓN (R1 acc@1_mcq) 

Implementa la evaluación baseline del conjunto R1 en formato MCQ (1 correcta + N distractores) usando el modelo causal. La puntuación de cada opción se calcula como la suma de log‑probabilidades de todos los sub‑tokens de la continuación dada el prompt. Además, se reportan métricas de sanity (P@k del primer sub‑token) para validar el comportamiento del siguiente token.

**Acc@1_mcq (porcentaje):**


$$
\mathrm{acc@1\_mcq} \;=\; \frac{100}{N}\sum_{i=1}^{N} \mathbf{1}\!\left[
\arg\max_{o \in O_i} s_i(o) \;=\; g_i
\right]
$$


donde:


- $N$: número de ejemplos del conjunto de evaluación.
- $O_i = \{g_i\} \cup D_i$: conjunto de opciones del ítem $i$ (la correcta $g_i$ más los distractores $D_i$).
- $g_i$: opción correcta (gold) del ítem $i$.
- $s_i(o)$: puntuación de la opción $o$ para el ítem $i$. Es decir, entre las 4 opciones (1 correcta + 3 distractores), calcula cuál es la más probable para el modelo y se queda con la de mayor puntuación.
- $\mathbf{1}[\cdot]$: **función indicadora** (vale 1 si la condición es verdadera; 0 en caso contrario).

* P@1_first_subtoken = el porcentaje de veces que el primer sub-token correcto es la opción #1 (la más probable) que el modelo pondría como siguiente token dado el prompt.

* P@5_first_subtoken = porcentaje de veces que ese primer sub-token correcto cae dentro del Top-5 más probables.

* P@10_first_subtoken = lo mismo pero dentro del Top-10.

O sea, no evalúa la respuesta completa, solo el primer pedacito (sub-token) que el modelo generaría a continuación. Es un chequeo rápido de que el modelo “apunta” bien al inicio de la respuesta.

In [ ]:
@torch.no_grad()
def seq_logprob(prompt: str, continuation: str) -> float:
    """
    Suma log-probs de TODOS los sub-tokens de la continuación, condicionados al prompt.
    Promueve evaluación robusta (no exige un solo token BPE).
    """
    prompt = prompt.rstrip()
    if not prompt.endswith(" "):
        prompt = prompt + " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(DEVICE)  # [1, T]
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]

    # Si la continuación tokeniza a 0 sub-tokens, devolvemos -inf para que nunca gane.
    if toks_all.shape[1] <= P:
        return float("-inf")

    logits = model(toks_all)                        # [1, T, V]
    logprobs = F.log_softmax(logits, dim=-1)       # [1, T, V]
    cont_ids = toks_all[0, P:]                     # [C]
    pref = logprobs[0, P-1:-1]                     # [C, V] prob del token t condicionado en t-1
    lp = pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()
    return lp

@torch.no_grad()
def first_subtoken_id(s: str) -> int | None:
    """
    Devuelve el ID del PRIMER sub-token de la cadena (con espacio inicial),
    o None si la tokenización devuelve 0 tokens.
    """
    t = model.to_tokens(ensure_space_prefix(s), prepend_bos=False)
    # t forma [1, T]; puede ser [1,0], [1,1], etc.
    if t.ndim == 2 and t.shape[1] > 0:
        return t[0, 0].item()
    if t.numel() > 0:
        # fallback ultra-defensivo (por si el backend cambia dims)
        return t.flatten()[0].item()
    return None

def evaluate_r1_baseline_mcq(df_eval: pd.DataFrame, num_distractors: int = 3):
    """
    MCQ: para cada (prompt, gold) muestrea distractores y elige por máxima log-prob de la continuación completa.
    También reporta P@k del primer sub-token como sanity (no es métrica oficial).
    """
    hits_mcq = 0
    hits_p1 = hits_p5 = hits_p10 = 0
    total = 0

    print("[INFO] Evaluando R1 (Baseline, sin intervención) ...")
    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
        prompt = row["prompt_cloze"]
        gold   = ensure_space_prefix(row["target_token"])

        options = [gold] + sample_distractors(gold, k=num_distractors)
        if not options:
            continue

        scores = [seq_logprob(prompt, opt) for opt in options]
        pred_idx = int(torch.tensor(scores).argmax().item())
        hits_mcq += int(options[pred_idx] == gold)

        # Sanity: P@k del primer sub-token (si aplica)
        with torch.no_grad():
            logits = model(prompt, return_type="logits")  # distribución del próximo sub-token
            last = logits[0, -1]
            gid = first_subtoken_id(gold)
            if gid is not None:
                top10 = torch.topk(last, k=10).indices.tolist()
                hits_p1  += int(gid == top10[0])
                hits_p5  += int(gid in top10[:5])
                hits_p10 += int(gid in top10[:10])
            # Si gid es None, simplemente no sumamos a los contadores de sanity

        total += 1

    den = max(total, 1)
    out = {
        "acc@1_mcq": 100 * hits_mcq / den,
        "P@1_first_subtoken": 100 * hits_p1 / den,
        "P@5_first_subtoken": 100 * hits_p5 / den,
        "P@10_first_subtoken": 100 * hits_p10 / den,
        "N": den,
    }
    return out

In [ ]:


# ---- Corre baseline ----
baseline_metrics = evaluate_r1_baseline_mcq(df, num_distractors=NUM_DISTRACTORS)
print("\n================= BASELINE R1 =================")
print(f"Total ejemplos         : {baseline_metrics['N']}")
print(f"acc@1_mcq              : {baseline_metrics['acc@1_mcq']:.2f}%")
print(f"P@1 (primer sub-token) : {baseline_metrics['P@1_first_subtoken']:.2f}%  [sanity]")
print(f"P@5 (primer sub-token) : {baseline_metrics['P@5_first_subtoken']:.2f}%  [sanity]")
print(f"P@10(primer sub-token) : {baseline_metrics['P@10_first_subtoken']:.2f}% [sanity]")
print("================================================\n")


[INFO] Cargando modelo 'gpt2-small' en cuda ...
Loaded pretrained model gpt2-small into HookedTransformer
[INFO] Modelo cargado.
[INFO] Cargando dataset R1 desde: Data/dataset_R1_en.csv
[INFO] Filtrado opcional: 348 -> 322 filas.
[INFO] Evaluando R1 (Baseline, sin intervención) ...


100%|██████████| 322/322 [00:22<00:00, 14.08it/s]


================= BASELINE R1 =================
Total ejemplos         : 322
acc@1_mcq              : 60.87%
P@1 (primer sub-token) : 11.80%  [sanity]
P@5 (primer sub-token) : 32.30%  [sanity]
P@10(primer sub-token) : 49.38% [sanity]



# Desarrollo con modelo GPT 2 Medium

Este bloque carga tokenizer y pesos de gpt2-medium desde disco (sin internet), crea el modelo de Hugging Face en CPU y fp32 (para evitar mezclas de device/dtype), y luego lo envuelve con HookedTransformer (TransformerLens), moviendo todo al dispositivo final a GPU  y a la precisión objetivo (fp16 por defecto). Finalmente asegura que el tokenizer usado por el wrapper sea el mismo.

1) Imports y entorno

* os.environ.setdefault: evita que transformers intente importar torchvision (no lo necesitamos y ahorra tiempo/ruido).

* AutoTokenizer, AutoModelForCausalLM: utilidades HF para cargar tokenizer y modelo causal.

* HookedTransformer: wrapper de TransformerLens para hacer hooks y análisis mecanístico.

2) Parámetros básicos

* MODEL_LOCAL_DIR = "./models/gpt2-medium": ruta donde descargaste config.json, tokenizer.json, merges.txt, vocab.json, model.safetensors.

* DTYPE_TARGET = torch.float16 if DEVICE=="cuda" else torch.float32: precisión final (fp16 en GPU; fp32 en CPU). [cargar primero en CPU/fp32 y después mover a GPU/fp16 vía TransformerLens evita errores como “Expected all tensors to be on the same device” y mezcla de dtypes.]
* TransformerLens convierte el state_dict al formato que espera, y ya lo deja en DEVICE con dtype deseado.

* El warning sobre “reduced precision” es normal si usas fp16 en GPU.

In [6]:

import os, torch
os.environ.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer

MODEL_LOCAL_DIR = "./models/gpt2-medium"   # carpeta que descargaste
MODEL_NAME      = "gpt2-medium"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE_TARGET    = torch.float16 if DEVICE == "cuda" else torch.float32  # precisión final

print(f"[INFO] Cargando tokenizer desde {MODEL_LOCAL_DIR} ...")
tok = AutoTokenizer.from_pretrained(
    MODEL_LOCAL_DIR, local_files_only=True, use_fast=True
)

print(f"[INFO] Cargando HF CausalLM (CPU, fp32) desde {MODEL_LOCAL_DIR} ...")
# MUY IMPORTANTE: mantenerlo en CPU y fp32 aquí
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_LOCAL_DIR,
    local_files_only=True,
    torch_dtype=torch.float32,     # cargar en fp32
    low_cpu_mem_usage=True,
)
# NO hagas hf_model.to(DEVICE) aquí

print("[INFO] Envolviendo con TransformerLens y moviendo al dispositivo final ...")
model = HookedTransformer.from_pretrained(
    model_name=MODEL_NAME,         # nombre “oficial” para el mapeo de cfg
    hf_model=hf_model,             # pasamos el modelo ya cargado (CPU)
    tokenizer=tok,                 # tokenizer local
    device=DEVICE,                 # a dónde lo quieres al final
    move_to_device=True,           # ← que TL lo mueva tras procesar el state_dict
    dtype=DTYPE_TARGET,            # ← y cambie a fp16 si hay GPU (o fp32 en CPU)
)

# por si acaso, fija el tokenizer en el wrapper
model.set_tokenizer(tok)

print(f"[INFO] Modelo listo: {model.cfg.model_name} en {DEVICE}, dtype={DTYPE_TARGET}")


[INFO] Cargando tokenizer desde ./models/gpt2-medium ...
[INFO] Cargando HF CausalLM (CPU, fp32) desde ./models/gpt2-medium ...
[INFO] Envolviendo con TransformerLens y moviendo al dispositivo final ...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: e3c5e26c-d2ca-4f91-81bb-cd0c24ee33f3)')' thrown while requesting HEAD https://huggingface.co/gpt2-medium/resolve/main/config.json
Retrying in 1s [Retry 1/5].


Loaded pretrained model gpt2-medium into HookedTransformer
[INFO] Modelo listo: gpt2-medium en cuda, dtype=torch.float16


## 0) Librerias

Este bloque prepara el entorno de ejecución: importa las librerías que usaremos  y ajusta la salida estándar y variables de entorno para que los logs aparezcan al instante y para evitar que transformers intente cargar torchvision (que no necesitamos).

In [7]:

import os, re, random, difflib, time, sys
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer

# ---- Salida sin buffer (por si el entorno la “traga”) ----
try:
    sys.stdout.reconfigure(line_buffering=True)  # Py3.7+
except Exception:
    pass
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")

'1'


## CONFIG

Este bloque concentra todas las variables de configuración que controlan:

1) dónde están los datos y el modelo
2) en qué dispositivo corre (GPU/CPU),
3) el tipo numérico usado
4) la reproducibilidad
5) cómo se evalúa y loguea R1.

In [8]:
R1_PATH         = "Data/dataset_R1_en.csv"
MODEL_NAME      = "gpt2-medium"          # nombre para el cfg en TransformerLens
MODEL_LOCAL_DIR = "./models/gpt2-medium" # carpeta local con pesos+tokenizer
DEVICE          = "cuda"
DTYPE_TARGET    = torch.float16
SEED            = 42

NUM_DISTRACTORS = 3
DO_SANITY_P_AT_K = True

# Frecuencia de logs (cada cuántos ejemplos imprime una línea)
PRINT_EVERY_PRETOK = 25
PRINT_EVERY_EVAL   = 25

BAD_SUBSTRINGS = {
    "thing", "object", "stuff", "device type", "capability", "position",
    "most often", "mostly", "kind of", "nickname", "person", "being",
    "animal", "comics character", "fictional", "concept", "common liquid",
    "non fluidlike", "set of", "linear unit", "platform", "place", "area",
    "color", "tone", "sound", "pitch", "material", "organic matter"
}

## Setup

Esta sección prepara el entorno de ejecución y define utilidades básicas que usaremos en toda la evaluación: comprobación de GPU, configuración de rendimiento, semillas para reproducibilidad y funciones helper para limpiar texto y extraer conceptos.

In [9]:
assert torch.cuda.is_available(), "No hay CUDA disponible."
torch.backends.cudnn.benchmark = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def ensure_space_prefix(tok: str) -> str:
    tok = str(tok).rstrip()
    return tok if tok.startswith(" ") else (" " + tok)

def normalize(s: str) -> str:
    return " ".join(str(s).lower().strip().split())

def looks_bad(s: str) -> bool:
    s_norm = normalize(s)
    return any(bad in s_norm for bad in BAD_SUBSTRINGS)

def too_similar(a: str, b: str, thr: float = 0.8) -> bool:
    return difflib.SequenceMatcher(None, a, b).ratio() >= thr

def extract_concept_from_prompt(prompt: str) -> str | None:
    p = prompt.strip()
    m = re.match(r"^(?:A|An|The)\s+([A-Za-z_]+)\b.*?\bis\s+(?:a|an)\b", p, flags=re.IGNORECASE)
    if m: return m.group(1).lower()
    m = re.match(r"^([A-Za-z_]+)\b.*?\bis\s+(?:a|an)\b", p, flags=re.IGNORECASE)
    if m: return m.group(1).lower()
    return None


## Carga del modelo

In [10]:
log(f"GPU: {torch.cuda.get_device_name(0)}")
log(f"Cargando tokenizer desde {MODEL_LOCAL_DIR} ...")
tok = AutoTokenizer.from_pretrained(MODEL_LOCAL_DIR, local_files_only=True, use_fast=True)

log(f"Cargando HF CausalLM (CPU, fp32) desde {MODEL_LOCAL_DIR} ...")
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_LOCAL_DIR,
    local_files_only=True,
    torch_dtype=torch.float32,      # CPU+fp32
    low_cpu_mem_usage=True,
)

log("Envolviendo con TransformerLens y moviendo a CUDA fp16 ...")
model = HookedTransformer.from_pretrained(
    model_name=MODEL_NAME,
    hf_model=hf_model,
    tokenizer=tok,
    device=DEVICE,
    move_to_device=True,
    dtype=DTYPE_TARGET,
)
model.set_tokenizer(tok)

p0 = next(model.parameters())
assert p0.device.type == "cuda", "El modelo no quedó en GPU."
log(f"Modelo listo: {model.cfg.model_name} en {p0.device}, dtype={p0.dtype}")

[07:46:41] GPU: NVIDIA GeForce RTX 3060 Laptop GPU
[07:46:41] Cargando tokenizer desde ./models/gpt2-medium ...
[07:46:41] Cargando HF CausalLM (CPU, fp32) desde ./models/gpt2-medium ...
[07:46:41] Envolviendo con TransformerLens y moviendo a CUDA fp16 ...


Loaded pretrained model gpt2-medium into HookedTransformer
[07:46:44] Modelo listo: gpt2-medium en cuda:0, dtype=torch.float16


## Dataset

In [ ]:
log(f"Cargando dataset R1: {R1_PATH}")
df = pd.read_csv(R1_PATH)

needed = {"prompt_cloze", "target_token"}
missing = needed - set(df.columns)
assert not missing, f"Faltan columnas: {missing}"

df["target_token"] = df["target_token"].astype(str).apply(ensure_space_prefix)

orig_n = len(df)
if "hypernym" in df.columns:
    df = df[~df["hypernym"].astype(str).apply(looks_bad)].copy()
df = df[~df["target_token"].astype(str).apply(looks_bad)].copy()
df = df.drop_duplicates(subset=["prompt_cloze", "target_token"]).reset_index(drop=True)
log(f"Filtrado opcional: {orig_n} -> {len(df)} filas.")

if "concept" not in df.columns:
    df["concept"] = df["prompt_cloze"].apply(extract_concept_from_prompt)

ALL_TARGETS = sorted(set(df["target_token"].tolist()), key=lambda x: x)

def sample_distractors(gold: str, k: int = NUM_DISTRACTORS):
    gold_norm = normalize(gold)
    cands = [t for t in ALL_TARGETS if normalize(t) != gold_norm and not too_similar(normalize(t), gold_norm)]
    if len(cands) < k:
        cands = [t for t in ALL_TARGETS if normalize(t) != gold_norm]
    k = min(k, len(cands))
    return random.sample(cands, k) if k > 0 else []

## PRE-TOKENIZACIÓN (con prints) 

In [ ]:
log("Pre-tokenizando prompts...")
prompt_tokens = []
t0 = time.time()
N = len(df)
for i, p in enumerate(df["prompt_cloze"].tolist(), start=1):
    prompt_tokens.append(model.to_tokens(p, prepend_bos=False))  # CPU
    if i % PRINT_EVERY_PRETOK == 0 or i == N:
        elapsed = time.time() - t0
        itps    = i / max(elapsed, 1e-6)
        pct     = 100.0 * i / N
        eta     = (N - i) / itps if itps > 0 else float("inf")
        log(f"[pretok] {i}/{N} ({pct:5.1f}%) | {itps:5.1f} it/s | ETA {eta:5.1f}s")


## EVALUACIÓN (con prints) 

In [ ]:
@torch.no_grad()
def first_subtoken_id(s: str) -> int | None:
    t = model.to_tokens(ensure_space_prefix(s), prepend_bos=False)
    if t.ndim == 2 and t.shape[1] > 0:
        return t[0, 0].item()
    if t.numel() > 0:
        return t.flatten()[0].item()
    return None

@torch.no_grad()
def seq_logprob_tokens(prompt_tokens_gpu: torch.Tensor, continuation: str) -> float:
    continuation  = ensure_space_prefix(continuation)
    cont_tok_cpu  = model.to_tokens(continuation, prepend_bos=False)     # [1, C] CPU
    toks_all      = torch.cat([prompt_tokens_gpu, cont_tok_cpu.to(prompt_tokens_gpu.device)], dim=1)
    P             = prompt_tokens_gpu.shape[1]
    if toks_all.shape[1] <= P:
        return float("-inf")
    logits        = model(toks_all)                       # [1, T, V] en GPU
    logprobs      = F.log_softmax(logits.float(), dim=-1) # fp32 por estabilidad
    cont_ids      = toks_all[0, P:]
    pref          = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0], device=DEVICE), cont_ids].sum().item()

def evaluate_r1_baseline_mcq(df_eval: pd.DataFrame, num_distractors: int = 3):
    hits_mcq = hits_p1 = hits_p5 = hits_p10 = 0
    N = len(df_eval)
    log("Evaluando R1 (Baseline, sin intervención) ...")
    t0 = time.time()

    for i, row in enumerate(df_eval.itertuples(index=False), start=1):
        prompt_tok_gpu = prompt_tokens[i-1].to(DEVICE, non_blocking=True)
        gold = ensure_space_prefix(row.target_token)

        options = [gold] + sample_distractors(gold, k=num_distractors)
        if options:
            scores   = [seq_logprob_tokens(prompt_tok_gpu, opt) for opt in options]
            pred_idx = int(torch.tensor(scores, device=DEVICE).argmax().item())
            hits_mcq += int(options[pred_idx] == gold)

        if DO_SANITY_P_AT_K:
            logits = model(prompt_tok_gpu)  # [1, P, V]
            last   = logits[0, -1]
            gid    = first_subtoken_id(gold)
            if gid is not None:
                top10 = torch.topk(last, k=10).indices.tolist()
                hits_p1  += int(gid == top10[0])
                hits_p5  += int(gid in top10[:5])
                hits_p10 += int(gid in top10[:10])

        if i % PRINT_EVERY_EVAL == 0 or i == N:
            torch.cuda.synchronize()
            elapsed = time.time() - t0
            itps    = i / max(elapsed, 1e-6)
            acc     = 100.0 * hits_mcq / i
            pct     = 100.0 * i / N
            eta     = (N - i) / itps if itps > 0 else float("inf")
            log(f"[eval]   {i}/{N} ({pct:5.1f}%) | acc@1_mcq {acc:5.2f}% | {itps:5.1f} it/s | ETA {eta:5.1f}s")

    den = max(N, 1)
    return {
        "acc@1_mcq":          100 * hits_mcq / den,
        "P@1_first_subtoken": 100 * hits_p1 / den,
        "P@5_first_subtoken": 100 * hits_p5 / den,
        "P@10_first_subtoken":100 * hits_p10 / den,
        "N": den,
    }

In [ ]:
metrics = evaluate_r1_baseline_mcq(df, num_distractors=NUM_DISTRACTORS)

print("\n================= BASELINE R1 =================")
print(f"Total ejemplos         : {metrics['N']}")
print(f"acc@1_mcq              : {metrics['acc@1_mcq']:.2f}%")
print(f"P@1 (primer sub-token) : {metrics['P@1_first_subtoken']:.2f}%  [sanity]")
print(f"P@5 (primer sub-token) : {metrics['P@5_first_subtoken']:.2f}%  [sanity]")
print(f"P@10(primer sub-token) : {metrics['P@10_first_subtoken']:.2f}% [sanity]")
print("================================================\n")


[19:39:24] GPU: NVIDIA GeForce RTX 3060 Laptop GPU
[19:39:24] Cargando tokenizer desde ./models/gpt2-medium ...


[19:39:24] Cargando HF CausalLM (CPU, fp32) desde ./models/gpt2-medium ...
[19:39:24] Envolviendo con TransformerLens y moviendo a CUDA fp16 ...


Loaded pretrained model gpt2-medium into HookedTransformer
[19:39:26] Modelo listo: gpt2-medium en cuda:0, dtype=torch.float16
[19:39:26] Cargando dataset R1: Data/dataset_R1_en.csv
[19:39:26] Filtrado opcional: 348 -> 322 filas.
[19:39:26] Pre-tokenizando prompts...
[19:39:26] [pretok] 25/322 (  7.8%) | 5563.9 it/s | ETA   0.1s
[19:39:26] [pretok] 50/322 ( 15.5%) | 5439.1 it/s | ETA   0.1s
[19:39:26] [pretok] 75/322 ( 23.3%) | 4977.8 it/s | ETA   0.0s
[19:39:26] [pretok] 100/322 ( 31.1%) | 5178.3 it/s | ETA   0.0s
[19:39:26] [pretok] 125/322 ( 38.8%) | 5318.4 it/s | ETA   0.0s
[19:39:26] [pretok] 150/322 ( 46.6%) | 5308.4 it/s | ETA   0.0s
[19:39:26] [pretok] 175/322 ( 54.3%) | 5270.8 it/s | ETA   0.0s
[19:39:26] [pretok] 200/322 ( 62.1%) | 5364.1 it/s | ETA   0.0s
[19:39:26] [pretok] 225/322 ( 69.9%) | 5416.1 it/s | ETA   0.0s
[19:39:26] [pretok] 250/322 ( 77.6%) | 5156.4 it/s | ETA   0.0s
[19:39:26] [pretok] 275/322 ( 85.4%) | 5194.9 it/s | ETA   0.0s
[19:39:26] [pretok] 300/322 ( 9

# Modelos modificados -> Ablaciones

# 1) configura logs

In [ ]:
import time, gc, sys, math, random
import torch
import torch.nn.functional as F
import pandas as pd


## 1.1) desactivacion de barrias tqdm

In [ ]:

# Desactiva cualquier barra previa y hace stdout inmediato
try:
    import os
    os.environ["PYTHONUNBUFFERED"] = "1"
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

def log(msg):
    print(time.strftime("[%H:%M:%S]"), msg, flush=True)


Garantiza que las continuaciones empiecen con un espacio y Devuelve el id del primer sub-token de s con el tokenizer del modelo

In [ ]:
# ==== 1) Utilidades que ya tenías ====
def ensure_space_prefix(tok: str) -> str:
    tok = str(tok).rstrip()
    return tok if tok.startswith(" ") else (" " + tok)

@torch.no_grad()
def first_subtoken_id(s: str) -> int | None:
    t = model.to_tokens(ensure_space_prefix(s), prepend_bos=False)
    if t.ndim == 2 and t.shape[1] > 0:
        return int(t[0,0].item())
    return None if t.numel()==0 else int(t.flatten()[0].item())


Calcula la log-probabilidad de la continuación dada un prompt:

* Asegura un espacio final en el prompt y un espacio inicial en la continuación.

* Tokeniza prompt+continuation y separa P = len(prompt_tokens).

* Corre el modelo una sola vez y toma logprobs de los tokens de la continuación.

* Usa pref = logprobs[0, P-1:-1] (prob del token t condicionada en el estado del paso anterior) y suma las log-prob de los ids de la continuación.

Devuelve -inf si no hay continuación (protección).

In [ ]:

@torch.no_grad()
def seq_logprob(prompt: str, continuation: str) -> float:
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]
    if toks_all.shape[1] <= P:
        return float("-inf")

    logits = model(toks_all)                        # [1, T, V]
    logprobs = F.log_softmax(logits, dim=-1)
    cont_ids = toks_all[0, P:]
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()


CAV (Concept Activation Vector)

* Tool_seed: Lista semilla de conceptos de “herramienta”.
*

In [ ]:
TOOL_SEED = {
    "hammer","screwdriver","wrench","pliers","saw","drill","chisel","axe","knife",
    "lathe","router","sander","grinder","welder","clamp","vise","mallet","trowel",
    "spade","shovel","ladder"
}

def guess_is_tool(row) -> bool:
    c = str(row.get("concept", "")).lower()
    h = str(row.get("hypernym", "")).lower() if "hypernym" in row else ""
    return (c in TOOL_SEED) or ("tool" in h) or ("instrument" in h)

def prepare_labels_for_cav(df_full: pd.DataFrame):
    df_l = df_full.copy()
    if "is_tool" not in df_l.columns:
        df_l["is_tool"] = df_l.apply(guess_is_tool, axis=1)
    df_l = df_l[~df_l["concept"].isna()].copy()

    pos = [str(c) for c in df_l.loc[df_l["is_tool"], "concept"].dropna().unique().tolist()]
    neg = [str(c) for c in df_l.loc[~df_l["is_tool"], "concept"].dropna().unique().tolist()]
    return pos, neg



* Corre model.run_with_cache y devuelve el vector de activación en blocks.{layer}.{hook_name} para un token.

* token_idx=-1 significa “último token de la secuencia” (muy común para medir la frase completa).



In [ ]:
@torch.no_grad()
def get_cache_for_layer_token(text: str, layer: int,
                              hook_name="hook_resid_pre", token_idx: int = -1) -> torch.Tensor:
    logits, cache = model.run_with_cache(text, remove_batch_dim=False)
    act = cache[f'blocks.{layer}.{hook_name}']  # [1, seq, d_model]
    if token_idx < 0:
        token_idx = act.shape[1] + token_idx
    return act[0, token_idx, :].detach()

@torch.no_grad()
def build_cav_toolness_nobar(df_full: pd.DataFrame,
                             layer: int,
                             hook_name: str = "hook_resid_pre",
                             template: str = "A {X} is a",
                             n_pos: int = 50,
                             n_neg: int = 50,
                             log_every: int = 10):
    log(f"[CAV] Preparando listas POS/NEG…")
    pos_concepts, neg_concepts = prepare_labels_for_cav(df_full)
    random.shuffle(pos_concepts); random.shuffle(neg_concepts)
    pos_concepts = pos_concepts[:n_pos]
    neg_concepts = neg_concepts[:n_neg]
    log(f"[CAV] POS={len(pos_concepts)}  NEG={len(neg_concepts)}")

    if len(pos_concepts)==0 or len(neg_concepts)==0:
        raise ValueError("No hay suficientes conceptos positivos/negativos para CAV.")

    acts_pos, acts_neg = [], []

    log(f"[CAV] Etapa 2/3 – activaciones POS  (layer={layer}, hook={hook_name})…")
    for j, c in enumerate(pos_concepts, 1):
        prompt = template.replace("{X}", c)
        acts_pos.append(get_cache_for_layer_token(prompt, layer, hook_name, token_idx=-1))
        if j % log_every == 0 or j == len(pos_concepts):
            log(f"[CAV] POS {j}/{len(pos_concepts)}")

    log(f"[CAV] Etapa 3/3 – activaciones NEG  (layer={layer}, hook={hook_name})…")
    for j, c in enumerate(neg_concepts, 1):
        prompt = template.replace("{X}", c)
        acts_neg.append(get_cache_for_layer_token(prompt, layer, hook_name, token_idx=-1))
        if j % log_every == 0 or j == len(neg_concepts):
            log(f"[CAV] NEG {j}/{len(neg_concepts)}")

    mu_pos = torch.stack(acts_pos, dim=0).mean(0)
    mu_neg = torch.stack(acts_neg, dim=0).mean(0)
    v = mu_pos - mu_neg
    v = v / (v.norm() + 1e-8)
    log("[CAV] Dirección normalizada OK.")
    return v.to(DEVICE)

## Hook de steering + evaluación MCQ

* Crea un hook que modifica la activación en una posición (token_idx):

 * mode="add": resta alpha * v̂ (sustracción directa).

 * mode="project": resta alpha * proj_{v̂}(h) (elimina/atenúa el componente en la dirección v̂).

* Devuelve una función para pasar a run_with_hooks.

In [ ]:

def make_steer_hook(v_dir: torch.Tensor, alpha: float, mode: str = "project", token_idx: int = -1):
    v_hat = v_dir / (v_dir.norm() + 1e-8)
    def hook_fn(h, hook):
        idx = token_idx if token_idx >= 0 else (h.shape[1] + token_idx)
        h_out = h.clone()
        h_slice = h_out[:, idx, :]
        if mode == "add":
            h_slice = h_slice - alpha * v_hat
        elif mode == "project":
            proj = (h_slice @ v_hat)[:, None] * v_hat
            h_slice = h_slice - alpha * proj
        else:
            raise ValueError("mode debe ser 'add' o 'project'")
        h_out[:, idx, :] = h_slice
        return h_out
    return hook_fn


Igual que seq_logprob, pero corre el forward con fwd_hooks aplicados

In [ ]:
@torch.no_grad()
def seq_logprob_with_hooks(prompt: str, continuation: str, fwd_hooks=None) -> float:
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)
    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]
    logits = model.run_with_hooks(toks_all, fwd_hooks=fwd_hooks, return_type="logits")
    logprobs = F.log_softmax(logits, dim=-1)
    cont_ids = toks_all[0, P:]
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()


Toma k alternativas del pool distintas al oro, con normalización básica (lower/espacios).

In [ ]:

def sample_distractors(gold: str, pool, k: int):
    gold_norm = " ".join(str(gold).lower().split())
    cands = [t for t in pool if " ".join(str(t).lower().split()) != gold_norm]
    k = min(k, len(cands))
    return random.sample(cands, k) if k>0 else []


# Evaluación principal  

* Arma el hook con CAV v_dir, alpha, mode, en un layer/hook_name específicos.

* Define el pool de opciones como los target_token del dataset.

* Muestra el dataset (sample_n) para hacerlo rápido.

* Para cada fila:

 1) Toma el prompt (usa prompt_effective si existe, si no prompt_cloze).

 2) Crea opciones: [gold] + distractores.

 3) Calcula scores = log-prob de cada opción bajo el steering.

 4) Predice la opción con score máximo y acumula acc@1.

* Loguea progreso cada log_every y devuelve la accuracy final.

In [ ]:

@torch.no_grad()
def eval_R1_with_steering_nobar(df_eval: pd.DataFrame,
                                layer: int,
                                v_dir: torch.Tensor,
                                alpha: float = 1.0,
                                mode: str = "project",
                                hook_name: str = "hook_resid_pre",
                                num_distractors: int = 3,
                                sample_n: int | None = 60,
                                log_every: int = 10) -> float:
    steer_hook = make_steer_hook(v_dir=v_dir, alpha=alpha, mode=mode, token_idx=-1)
    fwd = [(f'blocks.{layer}.{hook_name}', steer_hook)]

    # pool de opciones
    ALL_TARGETS = sorted(set(df_eval["target_token"].tolist()))
    hits_mcq, total = 0, 0

    if sample_n is not None:
        df_run = df_eval.sample(n=min(sample_n, len(df_eval)), random_state=42)
        log(f"[EVAL] Usando muestra de {len(df_run)} items.")
    else:
        df_run = df_eval
        log(f"[EVAL] Usando dataset completo: {len(df_run)} items.")

    t0 = time.time()
    for i, (_, row) in enumerate(df_run.iterrows(), 1):
        prompt = row["prompt_effective"] if "prompt_effective" in row else row["prompt_cloze"]
        gold   = ensure_space_prefix(row["target_token"])
        options = [gold] + sample_distractors(gold, ALL_TARGETS, num_distractors)
        if not options:
            continue
        scores = [seq_logprob_with_hooks(prompt, opt, fwd_hooks=fwd) for opt in options]
        pred = options[int(torch.tensor(scores).argmax().item())]
        hits_mcq += int(pred == gold)
        total += 1

        if i % log_every == 0 or i == len(df_run):
            elapsed = time.time() - t0
            log(f"[EVAL] {i}/{len(df_run)}  acc@1_mcq parc={100*hits_mcq/max(total,1):.2f}%  "
                f"elapsed={elapsed:.1f}s")

    acc = 100 * hits_mcq / max(total, 1)
    log(f"[EVAL] Done. acc@1_mcq={acc:.2f}%  total={total}")
    return acc


Prueba

In [45]:
# Parámetros base
HOOK_NAME = "hook_resid_pre"
MODE      = "project"     # project = borrar componente
n_layers  = int(getattr(model.cfg, "n_layers", 12))
L_MID     = n_layers // 2
alpha0    = 3.5

# 1) Construye CAV pequeño y visible (sin barras, con logs)
v_mid = build_cav_toolness_nobar(
    df, layer=L_MID, hook_name=HOOK_NAME,
    n_pos=50, n_neg=50,   # empieza pequeño para verificar
    template="A {X} is a",
    log_every=10
)

# 2) Evalúa con intervención sobre una muestra de 60 items (rápido)
acc_int = eval_R1_with_steering_nobar(
    df, layer=L_MID, v_dir=v_mid, alpha=alpha0, mode=MODE,
    hook_name=HOOK_NAME, num_distractors=NUM_DISTRACTORS,
    sample_n=60, log_every=10
)
print(f"[RESULT] L{L_MID}  α={alpha0}  acc@1_mcq={acc_int:.2f}%")


[23:19:43] [CAV] Preparando listas POS/NEG…
[23:19:43] [CAV] POS=50  NEG=50
[23:19:43] [CAV] Etapa 2/3 – activaciones POS  (layer=12, hook=hook_resid_pre)…


Exception ignored in: <function tqdm.__del__ at 0x713dda5fe170>
Traceback (most recent call last):
  File "/home/juan/anaconda3/envs/tesis_gpt2/lib/python3.10/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/juan/anaconda3/envs/tesis_gpt2/lib/python3.10/site-packages/tqdm/std.py", line 1267, in close
    if self.disable:
AttributeError: 'tqdm' object has no attribute 'disable'
Exception ignored in: <function tqdm.__del__ at 0x713dda5fe170>
Traceback (most recent call last):
  File "/home/juan/anaconda3/envs/tesis_gpt2/lib/python3.10/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/home/juan/anaconda3/envs/tesis_gpt2/lib/python3.10/site-packages/tqdm/notebook.py", line 273, in close
    if self.disable:
AttributeError: 'tqdm' object has no attribute 'disable'
Exception ignored in: <function tqdm.__del__ at 0x713dda5fe170>
Traceback (most recent call last):
  File "/home/juan/anaconda3/envs/tesis_gpt2/lib/python3.10/site-packages

[23:19:44] [CAV] POS 10/50
[23:19:44] [CAV] POS 20/50
[23:19:45] [CAV] POS 30/50
[23:19:45] [CAV] POS 40/50
[23:19:46] [CAV] POS 50/50
[23:19:46] [CAV] Etapa 3/3 – activaciones NEG  (layer=12, hook=hook_resid_pre)…
[23:19:46] [CAV] NEG 10/50
[23:19:47] [CAV] NEG 20/50
[23:19:47] [CAV] NEG 30/50
[23:19:48] [CAV] NEG 40/50
[23:19:48] [CAV] NEG 50/50
[23:19:48] [CAV] Dirección normalizada OK.
[23:19:48] [EVAL] Usando muestra de 60 items.
[23:19:50] [EVAL] 10/60  acc@1_mcq parc=60.00%  elapsed=1.3s
[23:19:51] [EVAL] 20/60  acc@1_mcq parc=65.00%  elapsed=2.5s
[23:19:52] [EVAL] 30/60  acc@1_mcq parc=60.00%  elapsed=3.8s
[23:19:53] [EVAL] 40/60  acc@1_mcq parc=65.00%  elapsed=5.0s
[23:19:55] [EVAL] 50/60  acc@1_mcq parc=68.00%  elapsed=6.3s
[23:19:56] [EVAL] 60/60  acc@1_mcq parc=66.67%  elapsed=7.6s
[23:19:56] [EVAL] Done. acc@1_mcq=66.67%  total=60
[RESULT] L12  α=3.5  acc@1_mcq=66.67%


---

Esta celda implementa feature-steering “por concepto” con span correcto: en vez de empujar/ablar “todo” o un único token, interviene exactamente los sub-tokens que condicionan la continuación y usa una dirección (vector) construida específicamente para el concepto.

1. Identificar, en una plantilla (p. ej. "A {X} is a"), el span de sub-tokens que corresponde al concepto X.

2. Construir una dirección unitaria v̂(concept) promediando activaciones de ese span en una capa y hook dados.

3. Al evaluar log P(continuation | prompt), aplicar un hook (modo project o add) solo en las posiciones que causan cada token de la continuación: el tramo [P-1, P-1+C) donde P=len(prompt_tokens) y C=len(continuation_tokens).

4. O bien usar la misma dirección del oro para todas las opciones (ablación orientada al gold), o calcular una dirección por opción (per-option).

In [ ]:
import re
import time
from functools import lru_cache
import torch
import torch.nn.functional as F


* Toma la plantilla, coloca el concepto y ubica la primera ocurrencia en texto plano.

* Convierte prefijo y concepto a longitudes en tokens, no en caracteres (clave para BPE).

* Devuelve índices (i_from, i_to) del concepto en coordenadas de tokens del string final text.

* Usa ensure_space_prefix(concept) para que la tokenización de modelos estilo GPT-2 case bien (los BPE suelen codificar espacios).

In [ ]:

# --- Utilidades de tokenización/segmento ---

def _to_tokens(txt: str):
    return model.to_tokens(txt, prepend_bos=False).to(DEVICE)

def _tok_len(txt: str) -> int:
    return _to_tokens(txt).shape[1]

def find_concept_token_span(template: str, concept: str):
    """
    Devuelve (i_from, i_to) índices de tokens para la aparición de 'concept' en template.replace('{X}', concept).
    Tomamos los tokens del prefijo antes del concepto y los tokens del concepto (con espacio).
    """
    text = template.replace("{X}", concept)
    # Partimos por la PRIMERA ocurrencia de concept (asumimos plantilla simple)
    idx = text.lower().find(concept.lower())
    if idx < 0:
        return None  # No encontrado
    prefix = text[:idx]
    # Longitudes en tokens
    len_prefix = _tok_len(prefix)
    # MUY IMPORTANTE: usar espacio-líder para que case con BPE de GPT-2
    len_concept = _tok_len(ensure_space_prefix(concept))
    return len_prefix, len_prefix + len_concept, text


## Extraccion del concepto antes de calcular el hook
Localiza el span de sub-tokens del concepto dentro de la plantilla.

Corre el modelo con run_with_cache y toma las activaciones en blocks.{layer}.{hook_name}.

Extrae solo las filas del span del concepto y las promedia → obtiene un vector v ∈ ℝ^{d_model}.

Lo normaliza y devuelve v̂ (dirección unitaria del concepto) junto con el text usado.

In [ ]:

@torch.no_grad()
def get_dir_for_concept(concept: str, layer: int, hook_name: str, template: str = "A {X} is a"):
    """
    Construye una dirección unitaria para el CONCEPTO puntual:
    - Saca cache en (layer, hook) para el texto "A {concept} is a"
    - Promedia las activaciones de TODOS los sub-tokens del concepto
    """
    span = find_concept_token_span(template, concept)
    if span is None:
        return None, None  # no se halló
    i_from, i_to, text = span
    # Cache
    logits, cache = model.run_with_cache(text, remove_batch_dim=False)
    act = cache[f'blocks.{layer}.{hook_name}']  # [1, T, d]
    slice_act = act[0, i_from:i_to, :]          # [Lspan, d] los tokens del concepto
    v = slice_act.mean(0)                       # [d]
    n = v.norm()
    if float(n) == 0.0:
        return None, text
    return (v / (n + 1e-8)).to(DEVICE), text




Recibe v_hat ya calculado y crea un hook que aplica la intervención (modo add o project) sobre un tramo de posiciones [i_from, i_to) en las activaciones.

In [ ]:
#--- Hook sobre un TRAMO de posiciones ---
def make_steer_hook_span(v_hat: torch.Tensor, alpha: float, mode: str, i_from: int, i_to: int):
    """
    Interviene h[:, i_from:i_to, :] (semi-cerrado [i_from, i_to)).
    mode='project' -> h' = h - alpha * <h, v_hat> v_hat
    mode='add'     -> h' = h - alpha * v_hat
    """
    def hook(h, hook):
        h_out = h.clone()
        sl = h_out[:, i_from:i_to, :]           # [B, Lspan, d]
        if mode == "add":
            sl = sl - alpha * v_hat
        elif mode == "project":
            proj = (sl @ v_hat)[..., None] * v_hat
            sl = sl - alpha * proj
        else:
            raise ValueError("mode debe ser 'add' o 'project'")
        h_out[:, i_from:i_to, :] = sl
        return h_out
    return hook

calcula la log-prob total de continuation dado prompt, pero aplicando un hook (tu “lobotomía/steering”) solo en el tramo de activaciones que realmente condiciona cada token de la continuación.

In [ ]:

@torch.no_grad()
def seq_logprob_with_vdir_over_span(prompt: str, continuation: str,
                                    layer: int, hook_name: str,
                                    v_hat: torch.Tensor, alpha: float, mode: str):
    """
    Suma log-probs de la continuación, aplicando el hook sobre el tramo que SÍ cuenta:
    posiciones [P-1, P-1+C) (exactamente las que condicionan cada token de la continuación).
    """
    prompt = prompt.rstrip()
    if not prompt.endswith(" "):
        prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = _to_tokens(prompt + continuation)
    P = _tok_len(prompt)
    C = toks_all.shape[1] - P
    if C <= 0:
        return float("-inf")

    i_from = P - 1
    i_to   = P - 1 + C

    steer = make_steer_hook_span(v_hat=v_hat, alpha=alpha, mode=mode, i_from=i_from, i_to=i_to)
    logits = model.run_with_hooks(toks_all,
                                  fwd_hooks=[(f'blocks.{layer}.{hook_name}', steer)],
                                  return_type="logits")
    logprobs = F.log_softmax(logits, dim=-1)
    cont_ids = toks_all[0, P:]          # ids de la continuación
    pref = logprobs[0, P-1:-1]          # posiciones que los condicionan
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()


cachea (con LRU) la dirección del concepto v_hat(concept, layer, hook) para no recalcularla en cada item.

In [ ]:
@lru_cache(maxsize=4096)
def cached_dir_for_concept(concept: str, layer: int, hook_name: str, template: str):
    v_hat, _txt = get_dir_for_concept(concept, layer, hook_name, template)
    return v_hat

Evalúa precisión MCQ (acc@1) en el dataset aplicando feature-steering por concepto

In [ ]:
def eval_R1_per_concept(df_eval,
                        layer: int,
                        alpha: float = 1.0,
                        mode: str = "project",
                        hook_name: str = "hook_resid_pre",
                        num_distractors: int = 3,
                        template: str = "A {X} is a",
                        per_option: bool = False,   # True: v_hat distinto para cada opción
                        sample_n: int | None = None,
                        log_every: int = 20):
    """
    - per_option=False: usa v_hat del concepto ORO (gold) para TODAS las opciones del ítem (ablación orientada al gold).
    - per_option=True : recalcula v_hat para cada opción (ablación específica por candidato).
    """
    n_rows = len(df_eval) if sample_n is None else min(sample_n, len(df_eval))
    hits, total = 0, 0

    t0 = time.time()
    for i, (_, row) in enumerate(df_eval.iloc[:n_rows].iterrows(), start=1):
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold   = ensure_space_prefix(row["target_token"])
        options = [gold] + sample_distractors(gold, k=num_distractors)
        if not options:
            continue

        # v_hat base (por gold) si NO es por-opción
        v_hat_gold = None
        if not per_option:
            v_hat_gold = cached_dir_for_concept(gold.strip(), layer, hook_name, template)
            if v_hat_gold is None:
                # si no pudimos construir dirección, seguimos sin intervenir en este item
                scores = [seq_logprob_with_vdir_over_span(prompt, opt, layer, hook_name,
                                                          torch.zeros(model.cfg.d_model, device=DEVICE),
                                                          0.0, mode)
                          for opt in options]
            else:
                scores = [seq_logprob_with_vdir_over_span(prompt, opt, layer, hook_name,
                                                          v_hat_gold, alpha, mode)
                          for opt in options]
        else:
            # per-opción: cada score usa su propia dirección v_hat(opción)
            scores = []
            for opt in options:
                v_hat_opt = cached_dir_for_concept(opt.strip(), layer, hook_name, template)
                if v_hat_opt is None:
                    # sin dirección -> sin intervención para esta opción
                    s = seq_logprob_with_vdir_over_span(prompt, opt, layer, hook_name,
                                                        torch.zeros(model.cfg.d_model, device=DEVICE),
                                                        0.0, mode)
                else:
                    s = seq_logprob_with_vdir_over_span(prompt, opt, layer, hook_name,
                                                        v_hat_opt, alpha, mode)
                scores.append(s)

        pred_idx = int(torch.tensor(scores).argmax().item())
        hits += int(options[pred_idx] == gold)
        total += 1

        if log_every and (i % log_every == 0 or i == n_rows):
            dt = time.time() - t0
            print(f"[EVAL/{i:>4}/{n_rows}] acc_parc={100*hits/max(total,1):5.2f}%  elapsed={dt:0.1f}s")

    return 100 * hits / max(total, 1)


Toma todos los target_token únicos del df, elimina nulos y los convierte a str.

A cada uno le antepone un espacio con ensure_space_prefix(...)


## Genera la lista de candidatos de donde se extraerán los distractores.

In [63]:
# 0) Pool global de candidatos (se construye una sola vez)
CAND_POOL = [
    ensure_space_prefix(str(t))
    for t in pd.Series(df["target_token"]).dropna().unique().tolist()
]
print(f"[POOL] candidatos={len(CAND_POOL)}")


[POOL] candidatos=57


## Muestreador 

In [ ]:
import random

def sample_distractors_safe(gold: str, k: int, pool=None):
    if pool is None:
        pool = CAND_POOL

    gold_norm = normalize(gold) if "normalize" in globals() else str(gold).strip().lower()
    cands = []
    for t in pool:
        t_norm = normalize(t) if "normalize" in globals() else str(t).strip().lower()
        if t_norm == gold_norm:
            continue
        if "too_similar" in globals() and too_similar(t_norm, gold_norm):
            continue
        cands.append(t)

    k = min(k, len(cands))
    return random.sample(cands, k) if k > 0 else []


 cualquier llamada antigua a sample_distractors ahora usa el pool por defecto

In [ ]:
def sample_distractors(gold: str, k: int, pool=None):
    return sample_distractors_safe(gold, k, pool=pool)


Construye la lista de opciones para el ítem de MCQ: el oro + NUM_DISTRACTORS distractores muestreados del pool.

In [67]:
options = [gold] + sample_distractors(gold, k=NUM_DISTRACTORS, pool=CAND_POOL)


In [68]:
HOOK_NAME = "hook_resid_pre"
MODE      = "project"
LAYER     = model.cfg.n_layers // 2
ALPHA     = 3.5
TEMPLATE  = "A {X} is a"

acc_gold = eval_R1_per_concept(
    df, layer=LAYER, alpha=ALPHA, mode=MODE,
    hook_name=HOOK_NAME, num_distractors=NUM_DISTRACTORS,
    template=TEMPLATE, per_option=False, sample_n=80, log_every=10
)
print(f"[RESULT] acc@1_mcq (v_hat del GOLD) = {acc_gold:0.2f}%")


[EVAL/  10/80] acc_parc=90.00%  elapsed=1.4s
[EVAL/  20/80] acc_parc=80.00%  elapsed=2.8s
[EVAL/  30/80] acc_parc=70.00%  elapsed=4.2s
[EVAL/  40/80] acc_parc=67.50%  elapsed=5.5s
[EVAL/  50/80] acc_parc=66.00%  elapsed=6.8s
[EVAL/  60/80] acc_parc=60.00%  elapsed=8.2s
[EVAL/  70/80] acc_parc=54.29%  elapsed=9.7s
[EVAL/  80/80] acc_parc=51.25%  elapsed=11.1s
[RESULT] acc@1_mcq (v_hat del GOLD) = 51.25%


# Calculo de Baseline

In [70]:
def eval_R1_nohooks(df_eval, num_distractors=3, sample_n=80, log_every=10):
    import time, torch, torch.nn.functional as F
    if isinstance(sample_n, int) and sample_n>0:
        df_eval = df_eval.sample(n=min(sample_n, len(df_eval)), random_state=0)
    hits=total=0; t0=time.time()
    for i,(_,row) in enumerate(df_eval.iterrows(),1):
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold   = ensure_space_prefix(row["target_token"])
        options = [gold] + sample_distractors_safe(gold, k=NUM_DISTRACTORS, pool=CAND_POOL)
        if not options: continue
        # logprob sin hooks
        prompt = prompt.rstrip()+" " if not prompt.endswith(" ") else prompt
        P = model.to_tokens(prompt, prepend_bos=False).shape[1]
        scores=[]
        for opt in options:
            toks_all = model.to_tokens(prompt+ensure_space_prefix(opt), prepend_bos=False).to(DEVICE)
            logits = model(toks_all)              # [1,T,V]
            logprobs = F.log_softmax(logits, dim=-1)
            cont_ids = toks_all[0,P:]
            pref = logprobs[0, P-1:-1]
            lp = pref[torch.arange(len(cont_ids)), cont_ids].sum().item()
            scores.append(lp)
        pred = options[int(torch.tensor(scores).argmax().item())]
        hits += int(pred==gold); total += 1
        if total%log_every==0:
            print(f"[BASE] {total:3d}/{len(df_eval):3d} acc@1={100*hits/total:5.2f}%  elapsed={time.time()-t0:0.1f}s")
    return 100*hits/max(total,1)

acc_base = eval_R1_nohooks(df, num_distractors=NUM_DISTRACTORS, sample_n=80)
print(f"[RESULT] baseline acc@1_mcq = {acc_base:0.2f}%")


[BASE]  10/ 80 acc@1=30.00%  elapsed=1.2s
[BASE]  20/ 80 acc@1=50.00%  elapsed=2.4s
[BASE]  30/ 80 acc@1=53.33%  elapsed=3.6s
[BASE]  40/ 80 acc@1=57.50%  elapsed=4.8s
[BASE]  50/ 80 acc@1=58.00%  elapsed=6.0s
[BASE]  60/ 80 acc@1=60.00%  elapsed=7.2s
[BASE]  70/ 80 acc@1=61.43%  elapsed=8.4s
[BASE]  80/ 80 acc@1=62.50%  elapsed=9.6s
[RESULT] baseline acc@1_mcq = 62.50%


In [75]:
@torch.no_grad()
def _seq_logprob_with_hooks(prompt: str, continuation: str, fwd_hooks=None) -> float:
    """
    Forward robusto:
      - Si fwd_hooks es None o vacío -> forward normal (sin hooks).
      - Si trae hooks -> run_with_hooks con lista (nunca None).
    """
    prompt = prompt.rstrip()
    if not prompt.endswith(" "):
        prompt = prompt + " "
    cont = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + cont, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]

    # --- FIX CLAVE: normal vs con hooks ---
    if not fwd_hooks:  # None o []
        logits = model(toks_all, return_type="logits")
    else:
        hooks_list = list(fwd_hooks)  # garantiza iterable
        logits = model.run_with_hooks(toks_all, fwd_hooks=hooks_list, return_type="logits")
    # --------------------------------------

    logprobs = F.log_softmax(logits.float(), dim=-1)
    cont_ids = toks_all[0, P:]
    pref = logprobs[0, P-1:-1]
    lp = pref[torch.arange(len(cont_ids), device=cont_ids.device), cont_ids].sum().item()
    return lp


# Grip de prueba para el analisis

In [ ]:
import time, math, random
import pandas as pd
import torch
import torch.nn.functional as F


# Variables globales para la prueba:
- HOOK_NAME       = "hook_resid_pre"
- MODE            = "project"                 # "project" (recomendado) o "add"
- LAYERS_GRID     = [model.cfg.n_layers//2]   # cambia a list(range(model.cfg.n_layers)) si quieres
- ALPHAS          = [3.5]
- NUM_DISTRACTORS = 3
- SAMPLE_N        = 80                        # cuantos ítems del df por corrida
- LOG_EVERY       = 10                        # cada cuantos ítems reportar progreso
- RESULTS_CSV     = "grid_results_verbose.csv"



In [ ]:

HOOK_NAME       = "hook_resid_pre"
MODE            = "project"                 # "project" (recomendado) o "add"
LAYERS_GRID     = [model.cfg.n_layers//2]   # cambia a list(range(model.cfg.n_layers)) si quieres
ALPHAS          = [3.5]
NUM_DISTRACTORS = 3
SAMPLE_N        = 80                        # cuantos ítems del df por corrida
LOG_EVERY       = 10                        # cada cuantos ítems reportar progreso
RESULTS_CSV     = "grid_results_verbose.csv"


# qué “formas” correr:

In [ ]:
FORMS = ["baseline", "gold_dir", "option_dir"]  
# - baseline  : sin intervención
# - gold_dir  : abla con v_hat del concepto “gold”
# - option_dir: abla con v_hat del concepto de cada opción


In [ ]:

# ---------- LOGGING UTILS ----------
def log(msg): 
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

class StageTimer:
    def __init__(self): self.t = time.time()
    def reset(self):     self.t = time.time()
    def elapsed(self):   return time.time() - self.t

# ---------- TEXT/TOK UTILS ----------
def ensure_space_prefix(s: str) -> str:
    s = str(s or "")
    return s if s.startswith(" ") else (" " + s)

def to_tok_ids(text: str):
    return model.to_tokens(text, prepend_bos=False).to(DEVICE)[0]

def find_subseq(hay_ids: torch.Tensor, nee_ids: torch.Tensor):
    H, N = len(hay_ids), len(nee_ids)
    if N == 0 or N > H: return None
    for i in range(H - N + 1):
        if torch.equal(hay_ids[i:i+N], nee_ids): return (i, i+N)
    return None

def get_act_for_positions(prompt_text: str, layer: int, hook_name: str, pos_indices):
    toks = model.to_tokens(prompt_text, prepend_bos=False).to(DEVICE)
    logits, cache = model.run_with_cache(toks, remove_batch_dim=False)
    H = cache[f'blocks.{layer}.{hook_name}']  # [1, seq, d_model]
    idx = torch.as_tensor(pos_indices, device=H.device)
    return H[0, idx, :].detach()


In [ ]:
# ---------- POOL/DICCIONARIOS DESDE DF ----------
GENERIC_HS = {"thing","object","device","device type","stuff","material","entity",
              "concept","type","kind","kind of","being","animal","person"}

def _norm_surface(s: str) -> str:
    return ensure_space_prefix(str(s or "").strip().lower())

def build_surface_maps(df_eval: pd.DataFrame):
    d = df_eval.copy()
    d["surface"]   = d["target_token"].astype(str).str.strip()
    d["surface_n"] = d["surface"].str.lower()
    d["concept"]   = d.get("concept", pd.Series([None]*len(d))).astype(str).str.strip()
    d["hypernym"]  = d.get("hypernym", pd.Series([None]*len(d))).astype(str).str.strip().str.lower()

    grp = d.groupby("surface_n")
    s2c = (grp["concept"].agg(lambda s: s.value_counts().idxmax() if len(s.dropna()) else None).to_dict())
    s2h = (grp["hypernym"].agg(lambda s: s.value_counts().idxmax() if len(s.dropna()) else None).to_dict())
    for k,h in list(s2h.items()):
        if (h is None) or (h in GENERIC_HS): s2h[k] = None

    pool = sorted({ensure_space_prefix(s) for s in d["surface"].unique().tolist()
                   if isinstance(s, str) and s.strip()})
    return s2c, s2h, pool

SURF2CONCEPT, SURF2HYPERNYM, CAND_POOL = build_surface_maps(df)

# ---------- DISTRACTORES SEGUROS ----------
def sample_distractors_safe(gold_surface: str, k: int, pool=None):
    pool = pool or CAND_POOL
    gold_norm = _norm_surface(gold_surface)
    cands = [p for p in pool if _norm_surface(p) != gold_norm]
    if not cands: return []
    k = min(k, len(cands))
    return random.sample(cands, k) if k>0 else []

# ---------- DIRECCIONES POR CONCEPTO ----------
_CONCEPT_CACHE = {}

@torch.no_grad()
def build_concept_dir(concept: str, hypernym: str|None, layer: int, hook_name: str):
    """
    v(concept): promedio de activaciones de los subtokens de 'concept'
    en plantillas simples; si hay hypernym útil añade "A {X} is a {H}".
    """
    key = (layer, hook_name, (concept or "").lower().strip(), (hypernym or None))
    if key in _CONCEPT_CACHE: return _CONCEPT_CACHE[key]

    X = (concept or "").strip()
    templates = ["A {X} is a", "The {X} is commonly used", "People often use the {X}", "This {X} is useful"]
    if hypernym and hypernym not in GENERIC_HS:
        templates.append(f"A {{X}} is a {hypernym}")

    x_ids = to_tok_ids(" " + X)
    acts = []
    for tpl in templates:
        prompt = tpl.replace("{X}", X)
        p_ids  = to_tok_ids(prompt)
        match  = find_subseq(p_ids, x_ids)
        if match is None:
            start, end = len(p_ids)-1, len(p_ids)
        else:
            start, end = match
        pos = list(range(start, end))
        a = get_act_for_positions(prompt, layer, hook_name, pos)  # [m,d]
        acts.append(a.mean(0))
    v = torch.stack(acts, dim=0).mean(0)
    v = v / (v.norm() + 1e-8)
    _CONCEPT_CACHE[key] = v
    return v

def make_ablate_hook(direction: torch.Tensor, alpha: float, mode: str, positions):
    v_hat = direction / (direction.norm() + 1e-8)
    pos_set = set(int(i) for i in positions)
    def hook_fn(h, hook):
        h_out = h.clone()
        idx   = torch.tensor(sorted(pos_set), device=h.device)
        h_sl  = h_out[:, idx, :]
        if mode == "add":
            h_sl = h_sl - alpha * v_hat
        elif mode == "project":
            proj = (h_sl @ v_hat)[..., None] * v_hat
            h_sl = h_sl - alpha * proj
        else:
            raise ValueError("mode debe ser 'project' o 'add'")
        h_out[:, idx, :] = h_sl
        return h_out
    return hook_fn

# ---------- LOGPROBS (con/sin hooks) ----------
@torch.no_grad()
def seq_logprob_nohooks(prompt: str, continuation: str) -> float:
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt+continuation, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]
    logits = model(toks_all)
    logprobs = F.log_softmax(logits, dim=-1)
    cont_ids = toks_all[0, P:]
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

@torch.no_grad()
def seq_logprob_ablate(prompt: str, option: str, layer: int, hook_name: str,
                       concept_vec: torch.Tensor, alpha: float, mode: str) -> float:
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    option = ensure_space_prefix(option)

    toks_all = model.to_tokens(prompt + option, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids = model.to_tokens(option, prepend_bos=False).to(DEVICE)[0]

    # posiciones exactas de la opción en la secuencia
    match = find_subseq(toks_all[0, P:], cont_ids)
    if match is None:
        opt_pos = [toks_all.shape[1]-1]
    else:
        a, b = match
        opt_pos = list(range(P + a, P + b))

    steer = make_ablate_hook(concept_vec, alpha, mode, opt_pos)
    logits = model.run_with_hooks(
        toks_all,
        fwd_hooks=[(f'blocks.{layer}.{hook_name}', steer)],
        return_type="logits"
    )
    logprobs = F.log_softmax(logits, dim=-1)
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

# ---------- EVALS ----------
def _concept_hyper_for_surface(surf: str):
    s = _norm_surface(surf)
    c = SURF2CONCEPT.get(s) or surf.strip()
    h = SURF2HYPERNYM.get(s)
    return c, h

@torch.no_grad()
def eval_baseline(df_eval: pd.DataFrame, num_distractors=3, sample_n=SAMPLE_N, log_every=LOG_EVERY):
    if isinstance(sample_n, int) and sample_n>0:
        df_eval = df_eval.sample(n=min(sample_n, len(df_eval)), random_state=0)
    hits = total = 0
    tmr = StageTimer()
    for _, row in df_eval.iterrows():
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold   = ensure_space_prefix(row["target_token"])
        options= [gold] + sample_distractors_safe(gold, k=num_distractors, pool=CAND_POOL)
        if not options: 
            continue
        scores = [seq_logprob_nohooks(prompt, opt) for opt in options]
        pred = options[int(torch.tensor(scores).argmax().item())]
        hits += int(pred==gold); total += 1
        if total % log_every == 0:
            acc = 100.0*hits/max(total,1)
            log(f"[BASELINE] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}%  elapsed={tmr.elapsed():0.1f}s")
    return 100.0*hits/max(total,1)

@torch.no_grad()
def eval_gold_dir(df_eval: pd.DataFrame, layer:int, alpha:float, mode:str,
                  hook_name:str, num_distractors=3, sample_n=SAMPLE_N, log_every=LOG_EVERY):
    if isinstance(sample_n, int) and sample_n>0:
        df_eval = df_eval.sample(n=min(sample_n, len(df_eval)), random_state=0)
    hits = total = 0; tmr = StageTimer()
    for _, row in df_eval.iterrows():
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold_s = ensure_space_prefix(row["target_token"])
        options= [gold_s] + sample_distractors_safe(gold_s, k=num_distractors, pool=CAND_POOL)
        if not options: 
            continue
        # v_hat del GOLD (concept/hypernym si existe)
        c,h = _concept_hyper_for_surface(gold_s)
        v   = build_concept_dir(c, h, layer, hook_name)
        scores = [seq_logprob_ablate(prompt, opt, layer, hook_name, v, alpha, mode) for opt in options]
        pred = options[int(torch.tensor(scores).argmax().item())]
        hits += int(pred==gold_s); total += 1
        if total % log_every == 0:
            acc = 100.0*hits/max(total,1)
            log(f"[GOLD_DIR   ] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}%  elapsed={tmr.elapsed():0.1f}s")
    return 100.0*hits/max(total,1)

@torch.no_grad()
def eval_option_dir(df_eval: pd.DataFrame, layer:int, alpha:float, mode:str,
                    hook_name:str, num_distractors=3, sample_n=SAMPLE_N, log_every=LOG_EVERY):
    if isinstance(sample_n, int) and sample_n>0:
        df_eval = df_eval.sample(n=min(sample_n, len(df_eval)), random_state=0)
    hits = total = 0; tmr = StageTimer()
    for _, row in df_eval.iterrows():
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold_s = ensure_space_prefix(row["target_token"])
        options= [gold_s] + sample_distractors_safe(gold_s, k=num_distractors, pool=CAND_POOL)
        if not options: 
            continue
        scores = []
        for opt_s in options:
            c,h = _concept_hyper_for_surface(opt_s)
            v   = build_concept_dir(c, h, layer, hook_name)
            scores.append(seq_logprob_ablate(prompt, opt_s, layer, hook_name, v, alpha, mode))
        pred = options[int(torch.tensor(scores).argmax().item())]
        hits += int(pred==gold_s); total += 1
        if total % log_every == 0:
            acc = 100.0*hits/max(total,1)
            log(f"[OPTION_DIR ] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}%  elapsed={tmr.elapsed():0.1f}s")
    return 100.0*hits/max(total,1)

In [ ]:


# ---------- GRID ----------
def run_grid_verbose(df_eval: pd.DataFrame, forms, layers, alphas):
    rows = []
    log("==== INICIO GRID ====")
    n_items = len(df_eval)
    log(f"Dataset: {n_items} items | sample_n={SAMPLE_N} | distractors={NUM_DISTRACTORS}")
    for form in forms:
        log(f"\n-- FORM: {form} --")
        for L in layers:
            for A in alphas:
                log(f"[GRID] L={L:02d} alpha={A} hook={HOOK_NAME} mode={MODE}")
                tmr = StageTimer()
                if form == "baseline":
                    acc = eval_baseline(df_eval, num_distractors=NUM_DISTRACTORS, sample_n=SAMPLE_N, log_every=LOG_EVERY)
                elif form == "gold_dir":
                    acc = eval_gold_dir(df_eval, L, A, MODE, HOOK_NAME, num_distractors=NUM_DISTRACTORS, sample_n=SAMPLE_N, log_every=LOG_EVERY)
                elif form == "option_dir":
                    acc = eval_option_dir(df_eval, L, A, MODE, HOOK_NAME, num_distractors=NUM_DISTRACTORS, sample_n=SAMPLE_N, log_every=LOG_EVERY)
                else:
                    raise ValueError(f"form desconocido: {form}")
                log(f"[RESULT] {form:11s} L={L:02d} α={A} acc@1={acc:0.2f}%  (t={tmr.elapsed():0.1f}s)")
                rows.append({"form":form, "layer":L, "alpha":A, "hook":HOOK_NAME, "mode":MODE, "acc@1":acc})
    log("\n==== FIN GRID ====\n")
    return pd.DataFrame(rows).sort_values(["form","layer","alpha"]).reset_index(drop=True)

# ---------- RUN ----------
torch.cuda.synchronize() if torch.cuda.is_available() else None
res_df = run_grid_verbose(df, FORMS, LAYERS_GRID, ALPHAS)
print("\n========= RESUMEN =========")
print(res_df.to_string(index=False))
res_df.to_csv(RESULTS_CSV, index=False)
log(f"Resultados guardados en {RESULTS_CSV}")


[00:27:10] ==== INICIO GRID ====
[00:27:10] Dataset: 322 items | sample_n=80 | distractors=3
[00:27:10] 
-- FORM: baseline --
[00:27:10] [GRID] L=12 alpha=3.5 hook=hook_resid_pre mode=project
[00:27:11] [BASELINE]  10/ 80 acc@1=50.00%  elapsed=1.2s
[00:27:12] [BASELINE]  20/ 80 acc@1=65.00%  elapsed=2.4s
[00:27:13] [BASELINE]  30/ 80 acc@1=63.33%  elapsed=3.6s
[00:27:15] [BASELINE]  40/ 80 acc@1=67.50%  elapsed=4.8s
[00:27:16] [BASELINE]  50/ 80 acc@1=66.00%  elapsed=6.0s
[00:27:17] [BASELINE]  60/ 80 acc@1=65.00%  elapsed=7.1s
[00:27:18] [BASELINE]  70/ 80 acc@1=64.29%  elapsed=8.3s
[00:27:19] [BASELINE]  80/ 80 acc@1=63.75%  elapsed=9.6s
[00:27:19] [RESULT] baseline    L=12 α=3.5 acc@1=63.75%  (t=9.6s)
[00:27:19] 
-- FORM: gold_dir --
[00:27:19] [GRID] L=12 alpha=3.5 hook=hook_resid_pre mode=project
[00:27:23] [GOLD_DIR   ]  10/ 80 acc@1=50.00%  elapsed=3.4s
[00:27:25] [GOLD_DIR   ]  20/ 80 acc@1=65.00%  elapsed=5.9s
[00:27:27] [GOLD_DIR   ]  30/ 80 acc@1=66.67%  elapsed=7.8s
[00:27:

In [79]:
# ===================== CONFIG =====================
import time, math, pandas as pd, torch
import torch.nn.functional as F

LOG_EVERY     = 20          # progreso cada N items
SAMPLE_N      = None        # None=usa todo df; o un int para subset comparable
NUM_DISTRACT  = 3
HOOKS         = ["hook_resid_pre"]  # puedes añadir otros (p.ej. "hook_resid_mid")
ALPHAS        = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 3.5, 5.0]
MODES         = ["project", "add"]  # project=borrar componente; add=resta fija
FORMS         = ["baseline", "gold_dir", "option_dir"]

# Capas a barrer: todas las del modelo
N_LAYERS      = int(getattr(model.cfg, "n_layers", 12))
LAYERS_GRID   = list(range(N_LAYERS))

def log(msg): 
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

# -------- helpers robustos --------
def _seq_logprob_with_hooks(prompt: str, continuation: str, fwd_hooks=None):
    """Log P(continuation | prompt), usando hooks si se pasan; safe contra None."""
    if fwd_hooks is None:
        fwd_hooks = []  # <--- evita TypeError con run_with_hooks
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids = model.to_tokens(continuation, prepend_bos=False).to(DEVICE)[0]

    logits = model.run_with_hooks(toks_all, fwd_hooks=fwd_hooks, return_type="logits")
    logprobs = F.log_softmax(logits, dim=-1)
    pref = logprobs[0, P-1:-1]
    lp = pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()
    return lp

@torch.no_grad()
def _eval_one_row_baseline(row, num_distractors):
    prompt = row.get("prompt_cloze") or row.get("prompt_effective")
    gold   = ensure_space_prefix(row["target_token"])
    options = [gold] + sample_distractors_safe(gold, k=num_distractors, pool=CAND_POOL)
    if not options:
        return None, None
    scores = [_seq_logprob_with_hooks(prompt, opt, fwd_hooks=[]) for opt in options]
    pred = options[int(torch.tensor(scores).argmax().item())]
    return pred, gold

@torch.no_grad()
def _eval_one_row_gold_dir(row, layer, hook_name, alpha, mode, num_distractors):
    prompt = row.get("prompt_cloze") or row.get("prompt_effective")
    gold_s = ensure_space_prefix(row["target_token"])
    options = [gold_s] + sample_distractors_safe(gold_s, k=num_distractors, pool=CAND_POOL)
    if not options:
        return None, None

    # dirección construida con el CONCEPTO del GOLD
    surf_n  = gold_s.strip().lower()
    concept = SURF2CONCEPT.get(surf_n) or gold_s.strip()
    hyper   = SURF2HYPERNYM.get(surf_n)
    d_vec   = build_concept_dir_conceptual(concept, hyper, layer=layer, hook_name=hook_name)

    scores = []
    for opt in options:
        lp = seq_logprob_ablate_option(prompt, opt, layer, hook_name, d_vec, alpha=alpha, mode=mode)
        scores.append(lp)
    pred = options[int(torch.tensor(scores).argmax().item())]
    return pred, gold_s

@torch.no_grad()
def _eval_one_row_option_dir(row, layer, hook_name, alpha, mode, num_distractors):
    prompt = row.get("prompt_cloze") or row.get("prompt_effective")
    gold_s = ensure_space_prefix(row["target_token"])
    options = [gold_s] + sample_distractors_safe(gold_s, k=num_distractors, pool=CAND_POOL)
    if not options:
        return None, None

    scores = []
    for opt in options:
        surf_n  = opt.strip().lower()
        concept = SURF2CONCEPT.get(surf_n) or opt.strip()
        hyper   = SURF2HYPERNYM.get(surf_n)
        d_vec   = build_concept_dir_conceptual(concept, hyper, layer=layer, hook_name=hook_name)
        lp      = seq_logprob_ablate_option(prompt, opt, layer, hook_name, d_vec, alpha=alpha, mode=mode)
        scores.append(lp)

    pred = options[int(torch.tensor(scores).argmax().item())]
    return pred, gold_s

# -------- evaluadores por forma --------
def eval_baseline(df_eval, num_distractors=NUM_DISTRACT, log_every=LOG_EVERY):
    hits = total = 0
    t0 = time.time()
    for _, row in df_eval.iterrows():
        pred, gold = _eval_one_row_baseline(row, num_distractors)
        if pred is None: 
            continue
        hits += int(pred == gold); total += 1
        if total % log_every == 0:
            acc = 100.0 * hits / max(total,1)
            log(f"[BASELINE] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}%  elapsed={time.time()-t0:0.1f}s")
    return 100.0 * hits / max(total,1)

def eval_gold_dir(df_eval, layer, alpha, mode, hook_name, num_distractors=NUM_DISTRACT, log_every=LOG_EVERY):
    hits = total = 0
    t0 = time.time()
    for _, row in df_eval.iterrows():
        pred, gold = _eval_one_row_gold_dir(row, layer, hook_name, alpha, mode, num_distractors)
        if pred is None: 
            continue
        hits += int(pred == gold); total += 1
        if total % log_every == 0:
            acc = 100.0 * hits / max(total,1)
            log(f"[GOLD_DIR   ] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}%  elapsed={time.time()-t0:0.1f}s")
    return 100.0 * hits / max(total,1)

def eval_option_dir(df_eval, layer, alpha, mode, hook_name, num_distractors=NUM_DISTRACT, log_every=LOG_EVERY):
    hits = total = 0
    t0 = time.time()
    for _, row in df_eval.iterrows():
        pred, gold = _eval_one_row_option_dir(row, layer, hook_name, alpha, mode, num_distractors)
        if pred is None: 
            continue
        hits += int(pred == gold); total += 1
        if total % log_every == 0:
            acc = 100.0 * hits / max(total,1)
            log(f"[OPTION_DIR ] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}%  elapsed={time.time()-t0:0.1f}s")
    return 100.0 * hits / max(total,1)

# -------- orquestador del GRID --------
def run_full_grid(df, forms=FORMS, layers=LAYERS_GRID, alphas=ALPHAS, modes=MODES, hooks=HOOKS,
                  num_distractors=NUM_DISTRACT, sample_n=SAMPLE_N, results_csv="grid_results_full.csv"):
    # Congelamos subset para comparabilidad
    if isinstance(sample_n, int) and sample_n > 0:
        df_run = df.sample(n=min(sample_n, len(df)), random_state=0)
    else:
        df_run = df.copy()

    log("==== INICIO GRID ====")
    log(f"Dataset: {len(df_run)} items | sample_n={sample_n if sample_n else 'all'} | distractors={num_distractors}")

    rows = []
    for hook_name in hooks:
        for mode in modes:
            for L in layers:
                for A in alphas:
                    for form in forms:
                        log(f"\n-- FORM: {form} --")
                        log(f"[GRID] L={L:02d} alpha={A} hook={hook_name} mode={mode}")
                        t0 = time.time()
                        if form == "baseline":
                            acc = eval_baseline(df_run, num_distractors=num_distractors, log_every=LOG_EVERY)
                        elif form == "gold_dir":
                            acc = eval_gold_dir(df_run, layer=L, alpha=A, mode=mode, hook_name=hook_name,
                                                num_distractors=num_distractors, log_every=LOG_EVERY)
                        elif form == "option_dir":
                            acc = eval_option_dir(df_run, layer=L, alpha=A, mode=mode, hook_name=hook_name,
                                                  num_distractors=num_distractors, log_every=LOG_EVERY)
                        else:
                            raise ValueError(f"Forma desconocida: {form}")

                        log(f"[RESULT] {form:11s} L={L:02d} α={A} acc@1={acc:0.2f}%  (t={time.time()-t0:0.1f}s)")
                        rows.append({
                            "form": form, "layer": L, "alpha": A, "hook": hook_name, "mode": mode, "acc@1": acc
                        })

    res_df = pd.DataFrame(rows).sort_values(["form","hook","mode","layer","alpha"]).reset_index(drop=True)
    print("\n========= RESUMEN =========")
    print(res_df.to_string(index=False))
    res_df.to_csv(results_csv, index=False)
    log(f"Resultados guardados en {results_csv}")
    log("==== FIN GRID ====")
    return res_df

# ===================== EJECUTAR =====================
results_df = run_full_grid(df,
                           forms=FORMS,
                           layers=LAYERS_GRID,
                           alphas=ALPHAS,
                           modes=MODES,
                           hooks=HOOKS,
                           num_distractors=NUM_DISTRACT,
                           sample_n=SAMPLE_N,                  # pon 80 como en tus pruebas, o None para todo
                           results_csv="grid_results_full.csv")


[00:32:54] ==== INICIO GRID ====
[00:32:54] Dataset: 322 items | sample_n=all | distractors=3
[00:32:54] 
-- FORM: baseline --
[00:32:54] [GRID] L=00 alpha=0.25 hook=hook_resid_pre mode=project
[00:32:56] [BASELINE]  20/322 acc@1=85.00%  elapsed=2.6s
[00:32:59] [BASELINE]  40/322 acc@1=87.50%  elapsed=5.1s
[00:33:01] [BASELINE]  60/322 acc@1=83.33%  elapsed=7.5s
[00:33:04] [BASELINE]  80/322 acc@1=75.00%  elapsed=10.0s
[00:33:06] [BASELINE] 100/322 acc@1=71.00%  elapsed=12.5s
[00:33:09] [BASELINE] 120/322 acc@1=71.67%  elapsed=15.0s
[00:33:11] [BASELINE] 140/322 acc@1=70.71%  elapsed=17.5s
[00:33:14] [BASELINE] 160/322 acc@1=70.00%  elapsed=20.0s
[00:33:16] [BASELINE] 180/322 acc@1=71.11%  elapsed=22.5s
[00:33:19] [BASELINE] 200/322 acc@1=70.00%  elapsed=25.0s
[00:33:21] [BASELINE] 220/322 acc@1=68.18%  elapsed=27.6s
[00:33:24] [BASELINE] 240/322 acc@1=67.50%  elapsed=30.1s
[00:33:27] [BASELINE] 260/322 acc@1=68.08%  elapsed=32.6s
[00:33:29] [BASELINE] 280/322 acc@1=68.21%  elapsed=35.

KeyboardInterrupt: 

In [97]:
# ===== PRUEBA CLARA DE ABLACIÓN (DEMO) =====
import time, pandas as pd, torch
import torch.nn.functional as F
SEED = 0
torch.manual_seed(SEED)

# --- Config demo (metodología) ---
ALPHA_DEMO   = 3.5            # usa también 5.0 si quieres ver efecto más fuerte
MODE_DEMO    = "project"      # 'project' = proyección ortogonal (borrar componente)
HOOKS_DEMO   = ["hook_resid_pre", "hook_resid_mid", "hook_mlp_out"]
N_LAYERS     = int(getattr(model.cfg, "n_layers", 24))
MID          = N_LAYERS // 2  # capas medias
LAYER_WIN    = 2              # L-2..L+2
K_NEXT       = 3              # P-1 + primeros k + toda la continuación
DEVICE       = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# --- Utiles mínimos (no pisan si ya existen) ---
if 'ensure_space_prefix' not in globals():
    def ensure_space_prefix(s: str) -> str:
        return s if s.startswith(" ") else " " + s

if 'sample_distractors_safe' not in globals():
    def sample_distractors_safe(gold, k=3, pool=None):
        import random
        pool = list(pool or [])
        pool = [w for w in pool if w.strip().lower() != gold.strip().lower()]
        random.seed(SEED)
        return random.sample(pool, min(k, len(pool)))

def _seq_lp_with_hooks(prompt: str, continuation: str, fwd_hooks=None):
    if fwd_hooks is None: fwd_hooks = []
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(DEVICE)
    P        = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids = model.to_tokens(continuation, prepend_bos=False).to(DEVICE)[0]

    logits = model.run_with_hooks(toks_all, fwd_hooks=fwd_hooks, return_type="logits")
    lp = F.log_softmax(logits, dim=-1)[0, P-1:-1]
    return lp[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

def _layer_window(center_L: int, width: int, n_layers: int):
    lo = max(0, center_L - width); hi = min(n_layers - 1, center_L + width)
    return list(range(lo, hi + 1))

def _positions_for_stepwise(toks_all, P, cont_ids, k_next=3, include_p_minus1=True):
    pos = set()
    if include_p_minus1 and P-1 >= 0: pos.add(P-1)
    end = min(P + k_next, toks_all.shape[1])
    pos.update(range(P, end))
    pos.update(range(P, toks_all.shape[1]))
    return sorted(pos)

def _make_multi_layer_multi_hook(direction, alpha, mode, positions, layers, hook_bundle):
    v_hat = direction / (direction.norm() + 1e-8)
    pos_idx = None
    def _mk_hook():
        def hook_fn(h, hook):
            nonlocal pos_idx
            h_out = h.clone()
            if pos_idx is None or pos_idx.device != h.device:
                pos_idx = torch.tensor(positions, device=h.device, dtype=torch.long)
            sl = h_out[:, pos_idx, :]
            if mode == "add":
                sl = sl - alpha * v_hat
            elif mode == "project":
                proj = (sl @ v_hat)[..., None] * v_hat
                sl = sl - alpha * proj
            else:
                raise ValueError("mode debe ser 'project' o 'add'")
            h_out[:, pos_idx, :] = sl
            return h_out
        return hook_fn
    fwd_hooks = []
    for L in layers:
        for hk in hook_bundle:
            fwd_hooks.append((f"blocks.{L}.{hk}", _mk_hook()))
    return fwd_hooks

def _concept_dir(surface: str, layer: int, hook_name: str):
    s = surface.strip().lower()
    concept = SURF2CONCEPT.get(s) or surface.strip()
    hyper   = SURF2HYPERNYM.get(s)
    return build_concept_dir_conceptual(concept, hyper, layer=layer, hook_name=hook_name)

def _contrastive_dir(gold_surface: str, distract_surfs, layer: int, hook_name: str):
    try:
        g  = _concept_dir(gold_surface, layer, hook_name)
        Ds = [ _concept_dir(ds, layer, hook_name) for ds in distract_surfs ] or [g]
        d  = g - torch.stack(Ds, 0).mean(0)
        return d / (d.norm() + 1e-8)
    except Exception:
        return _concept_dir(gold_surface, layer, hook_name)

# --- Baseline para 1 fila ---
def _score_row_baseline(row, k=3):
    prompt = row.get("prompt_cloze") or row.get("prompt_effective")
    gold   = ensure_space_prefix(row["target_token"])
    distract = sample_distractors_safe(gold, k=k, pool=globals().get("CAND_POOL"))
    options = [gold] + distract
    scores  = [_seq_lp_with_hooks(prompt, opt, []) for opt in options]
    best_i  = int(torch.tensor(scores).argmax().item())
    margin  = scores[0] - max(scores[1:]) if len(scores) > 1 else 0.0
    return {"options": options, "scores": scores, "pred": options[best_i], "margin": margin}

# --- Demo principal: muestra caída clara ---
def demo_ablation_effect(df, n=60, alpha=ALPHA_DEMO, mode=MODE_DEMO,
                         center_layer=MID, hook_bundle=HOOKS_DEMO,
                         layer_win=LAYER_WIN, k_next=K_NEXT):
    t0 = time.time()
    rows = []
    # 1) calculamos baseline en todo y elegimos n casos donde baseline acierta
    tmp = []
    for _, r in df.iterrows():
        b = _score_row_baseline(r, k=3)
        gold = ensure_space_prefix(r["target_token"])
        tmp.append((r, b))
    # quedarnos con correctos y con margen alto (para que el efecto se vea)
    correct = [(r,b) for (r,b) in tmp if b["pred"] == ensure_space_prefix(r["target_token"])]
    correct.sort(key=lambda t: t[1]["margin"], reverse=True)
    correct = correct[:min(n, len(correct))]

    # 2) ablar SOLO el gold con multi-capa + multi-hook + P-1+k+cont
    flips = 0
    deltas = []
    for r,b in correct:
        prompt = r.get("prompt_cloze") or r.get("prompt_effective")
        gold   = ensure_space_prefix(r["target_token"])
        distract = b["options"][1:]
        layers = _layer_window(center_layer, layer_win, N_LAYERS)
        d_vec  = _contrastive_dir(gold, distract, center_layer, "hook_resid_pre")

        toks_all = model.to_tokens((prompt.rstrip()+" ")+gold, prepend_bos=False).to(DEVICE)
        P        = model.to_tokens(prompt.rstrip()+" ", prepend_bos=False).shape[1]
        cont_ids = model.to_tokens(gold, prepend_bos=False).to(DEVICE)[0]
        pos      = _positions_for_stepwise(toks_all, P, cont_ids, k_next=k_next, include_p_minus1=True)
        fwd_hooks = _make_multi_layer_multi_hook(d_vec, alpha, mode, pos, layers, hook_bundle)

        # puntuar opciones: gold ablado, distractores intactos
        scores_abl = []
        for opt in b["options"]:
            if opt == gold:
                lp = _seq_lp_with_hooks(prompt, opt, fwd_hooks=fwd_hooks)
            else:
                lp = _seq_lp_with_hooks(prompt, opt, fwd_hooks=[])
            scores_abl.append(lp)

        best_i_abl = int(torch.tensor(scores_abl).argmax().item())
        pred_abl   = b["options"][best_i_abl]
        margin_abl = scores_abl[0] - max(scores_abl[1:]) if len(scores_abl) > 1 else 0.0
        flipped    = (pred_abl != gold)
        flips     += int(flipped)
        deltas.append(margin_abl - b["margin"])

        rows.append({
            "prompt": prompt[:120] + ("..." if len(prompt)>120 else ""),
            "gold": gold.strip(),
            "baseline_margin": round(b["margin"], 3),
            "abl_margin": round(margin_abl, 3),
            "delta_margin": round(margin_abl - b["margin"], 3),
            "flipped": flipped
        })

    df_demo = pd.DataFrame(rows)
    base_acc = 100.0 # por construcción (subset de correctos)
    post_acc = 100.0 * (len(df_demo) - flips) / max(1, len(df_demo))
    print("\n===== DEMO ABLATION =====")
    print(f"Casos evaluados: {len(df_demo)} (todos correctos en baseline)")
    print(f"Modo={mode}  α={alpha}  L={center_layer}±{layer_win}  hooks={'+'.join(hook_bundle)}")
    print(f"Baseline acc: {base_acc:0.2f}%  |  Post-ablación acc: {post_acc:0.2f}%  |  Flips: {flips}")
    if deltas:
        import numpy as np
        dm = np.array(deltas)
        print(f"Δ margen (abl - base): mean={dm.mean():.3f}  p50={np.median(dm):.3f}  p90={np.percentile(dm,90):.3f}")
    print(f"(t={time.time()-t0:0.1f}s)\n")
    # Mostrar los 5 casos con mayor caída de margen
    if not df_demo.empty:
        display(df_demo.sort_values("delta_margin").head(5))
    return df_demo

# ---- EJECUTAR DEMO ----
demo_df = demo_ablation_effect(
    df, n=60, alpha=ALPHA_DEMO, mode=MODE_DEMO,
    center_layer=MID, hook_bundle=HOOKS_DEMO, layer_win=LAYER_WIN, k_next=K_NEXT
)



===== DEMO ABLATION =====
Casos evaluados: 60 (todos correctos en baseline)
Modo=project  α=3.5  L=12±2  hooks=hook_resid_pre+hook_resid_mid+hook_mlp_out
Baseline acc: 100.00%  |  Post-ablación acc: 36.67%  |  Flips: 38
Δ margen (abl - base): mean=nan  p50=nan  p90=nan
(t=446.2s)



,prompt,gold,baseline_margin,abl_margin,delta_margin,flipped
32,A battery is a,electrical component,5.875,-25.594,-31.469,True
12,An inverter is a,electrical component,7.766,-22.797,-30.562,True
25,A fuse is a,electrical component,6.375,-22.906,-29.281,True
10,A plug is a,electrical device,8.203,-16.797,-25.000,True
1,A dehumidifier is a,electrical device,10.453,-13.969,-24.422,True


In [8]:
# ===================== LOBOTOMÍA – DOE METODOLÓGICO =====================
# Requiere: model (TransformerLens HookedTransformer) y df con columnas:
#   - target_token (str), prompt_cloze o prompt_effective (str)
#   - concept, hypernym (opcionales)
# Salidas: CSV incremental + subset S de baseline

import os, json, time, math, random
from typing import List, Tuple, Optional, Dict
import pandas as pd
import torch
import torch.nn.functional as F

# ---------- Config principal (metodología) ----------
SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

# Intensidades (metodología): 1.5 (baja), 3.5 (media), 6.0 (alta)
ALPHAS = [1.5, 3.5, 6.0]   # (H1C: dosis-respuesta)

# Localizaciones (metodología): capa media (50%) y tardía (75%)
def _layer_index(frac: float) -> int:
    nL = int(getattr(model.cfg, "n_layers", 12))
    # clamp y redondeo a entero de 0..nL-1
    return max(0, min(nL-1, int(round((nL-1) * frac))))

LAYERS_BY_LOC = {
    "mid":  _layer_index(0.50),
    "late": _layer_index(0.75),
}

# Técnica de ablación (factor A): sustracción vs proyección
# - add     => a' = a0 - α d          (sustracción)
# - project => a' = a0 - α·proj_d(a0)  (escalamos la proyección por α; α=1 es borrado “perfecto”)
MODES = ["add", "project"]

# Tipos de hook (factor adicional). Metodológicamente basta resid_pre; los otros son exploratorios.
HOOKS = ["hook_resid_pre"]            # puedes ampliar: ["hook_resid_pre","hook_mlp_out","hook_attn_out"]

# Distractores y logging
NUM_DISTRACT   = 3
LOG_EVERY      = 20

# Archivos de salida (checkpoint/reanudable)
RESULTS_CSV    = "grid_results_method.csv"
SUBSET_S_JSON  = "subset_S_indices.json"
SUBSET_S_CSV   = "subset_S_preview.csv"

# ---------- Utilidades base ----------
def log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

try:
    DEVICE
except NameError:
    try:
        DEVICE = next(model.parameters()).device
    except Exception:
        DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def ensure_space_prefix(s: str) -> str:
    s = str(s or "")
    return s if (len(s) > 0 and s[0].isspace()) else (" " + s)

# Construcción de mapas surface→concept/hypernym y pool de distractores
GENERIC_HS = {"thing","object","device","device type","stuff","material","entity",
              "concept","type","kind","kind of","being","animal","person"}

def build_surface_maps(df_in: pd.DataFrame):
    d = df_in.copy()
    d["surface"]   = d["target_token"].astype(str).str.strip()
    d["surface_n"] = d["surface"].str.lower()

    d["concept"]  = d.get("concept", pd.Series([None]*len(d))).astype(str).str.strip()
    d["hypernym"] = d.get("hypernym", pd.Series([None]*len(d))).astype(str).str.strip().str.lower()

    surf_grp = d.groupby("surface_n")
    surf2concept  = (surf_grp["concept"]
                     .agg(lambda s: s.value_counts().idxmax() if len(s.dropna()) else None)
                     .to_dict())
    surf2hypernym = (surf_grp["hypernym"]
                     .agg(lambda s: s.value_counts().idxmax() if len(s.dropna()) else None)
                     .to_dict())

    for k, h in list(surf2hypernym.items()):
        if (h is None) or (h in GENERIC_HS):
            surf2hypernym[k] = None

    cand_pool = sorted({
        ensure_space_prefix(s)
        for s in d["surface"].unique().tolist()
        if isinstance(s, str) and s.strip()
    })
    return surf2concept, surf2hypernym, cand_pool

SURF2CONCEPT, SURF2HYPERNYM, CAND_POOL = build_surface_maps(df)

def sample_distractors_safe(gold_surface: str, k: int, pool: List[str]):
    gold_norm = str(gold_surface or "").strip().lower()
    cand = [w for w in pool if str(w).strip().lower() != gold_norm]
    if len(cand) <= k:
        return cand
    rnd = random.Random(SEED + hash(gold_norm) % (10**6))
    return rnd.sample(cand, k)

# ---------- Tokenización/activaciones ----------
def to_tok_ids(text: str) -> torch.Tensor:
    return model.to_tokens(text, prepend_bos=False).to(DEVICE)[0]

def find_subseq(haystack_ids: torch.Tensor, needle_ids: torch.Tensor):
    H, N = len(haystack_ids), len(needle_ids)
    if N == 0 or N > H: return None
    for i in range(H - N + 1):
        if torch.equal(haystack_ids[i:i+N], needle_ids):
            return (i, i+N)
    return None

def get_act_for_positions(prompt_text: str, layer: int, hook_name: str, pos_indices):
    toks = model.to_tokens(prompt_text, prepend_bos=False).to(DEVICE)
    _, cache = model.run_with_cache(toks, remove_batch_dim=False)
    H = cache[f'blocks.{layer}.{hook_name}']  # [1, seq, d_model]
    idx = torch.as_tensor(list(pos_indices), device=H.device)
    return H[0, idx, :].detach()

# ---------- Dirección “conceptual” (concepto + hiperónimo opcional) ----------
_CONCEPT_CACHE: Dict = {}

@torch.no_grad()
def build_concept_dir_conceptual(concept: str,
                                 hypernym: Optional[str],
                                 layer: int,
                                 hook_name: str = "hook_resid_pre",
                                 base_templates = (
                                     "A {X} is a",
                                     "The {X} is commonly used",
                                     "People often use the {X}",
                                     "This {X} is useful",
                                 )) -> torch.Tensor:
    key = (layer, hook_name, (concept or "").lower().strip(), (hypernym or None))
    if key in _CONCEPT_CACHE:
        return _CONCEPT_CACHE[key]

    X = (concept or "").strip()
    if not X:
        raise ValueError("concept vacío en build_concept_dir_conceptual")

    templates = list(base_templates)
    if hypernym and hypernym not in GENERIC_HS:
        templates.append(f"A {{X}} is a {hypernym}")

    x_ids = to_tok_ids(" " + X)
    acts = []
    for tpl in templates:
        prompt = tpl.replace("{X}", X)
        p_ids  = to_tok_ids(prompt)
        match  = find_subseq(p_ids, x_ids)
        if match is None:
            start, end = len(p_ids)-1, len(p_ids)
        else:
            start, end = match
        pos = list(range(start, end))
        a = get_act_for_positions(prompt, layer, hook_name, pos)  # [m, d]
        acts.append(a.mean(0))
    d = torch.stack(acts, dim=0).mean(0)
    d = d / (d.norm() + 1e-8)
    _CONCEPT_CACHE[key] = d
    return d

# ---------- Hooks de ablación ----------
def make_ablate_hook(direction: torch.Tensor, alpha: float, mode: str, positions):
    v_hat = direction / (direction.norm() + 1e-8)
    pos_idx = torch.tensor(sorted({int(i) for i in positions}))

    def hook_fn(h, hook):
        v = v_hat.to(h.device)
        idx = pos_idx.to(h.device)
        h_out = h.clone()
        sl = h_out[:, idx, :]
        if mode == "add":  # sustracción
            sl = sl - alpha * v
        elif mode == "project":  # proyección (posible sobre/under con α)
            proj = (sl @ v)[..., None] * v
            sl = sl - alpha * proj
        else:
            raise ValueError("mode debe ser 'add' o 'project'")
        h_out[:, idx, :] = sl
        return h_out
    return hook_fn

# ---------- LogProb genérico con hooks ----------
@torch.no_grad()
def _seq_logprob_with_hooks(prompt: str, continuation: str, fwd_hooks=None):
    if fwd_hooks is None: fwd_hooks = []
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids = model.to_tokens(continuation, prepend_bos=False).to(DEVICE)[0]

    logits = model.run_with_hooks(toks_all, fwd_hooks=fwd_hooks, return_type="logits")
    logprobs = F.log_softmax(logits, dim=-1)
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

# Ablar SOLO los subtokens de la opción
@torch.no_grad()
def seq_logprob_ablate_option(prompt: str, option: str,
                              layer: int, hook_name: str,
                              direction: torch.Tensor,
                              alpha: float, mode: str):
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    option = ensure_space_prefix(option)

    toks_all  = model.to_tokens(prompt + option, prepend_bos=False).to(DEVICE)
    P         = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids  = model.to_tokens(option, prepend_bos=False).to(DEVICE)[0]

    match = find_subseq(toks_all[0, P:], cont_ids)
    if match is None:
        opt_pos = [toks_all.shape[1]-1]
    else:
        a, b   = match
        opt_pos= list(range(P + a, P + b))

    steer = make_ablate_hook(direction, alpha, mode, opt_pos)
    logits = model.run_with_hooks(toks_all, fwd_hooks=[(f'blocks.{layer}.{hook_name}', steer)],
                                  return_type="logits")
    logprobs = F.log_softmax(logits, dim=-1)
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

# Ablar TODA la continuación
@torch.no_grad()
def seq_logprob_ablate_allpos(prompt: str, option: str,
                              layer: int, hook_name: str,
                              direction: torch.Tensor,
                              alpha: float, mode: str):
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    option = ensure_space_prefix(option)

    toks_all  = model.to_tokens(prompt + option, prepend_bos=False).to(DEVICE)
    P         = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids  = model.to_tokens(option, prepend_bos=False).to(DEVICE)[0]

    opt_pos   = list(range(P, toks_all.shape[1]))
    steer     = make_ablate_hook(direction, alpha, mode, opt_pos)

    logits    = model.run_with_hooks(toks_all, fwd_hooks=[(f'blocks.{layer}.{hook_name}', steer)],
                                     return_type="logits")
    logprobs  = F.log_softmax(logits, dim=-1)
    pref      = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

# ---------- Baseline y subset S ----------
@torch.no_grad()
def _eval_one_row_baseline(row, num_distractors=NUM_DISTRACT):
    prompt = row.get("prompt_cloze") or row.get("prompt_effective")
    gold   = ensure_space_prefix(row["target_token"])
    opts   = [gold] + sample_distractors_safe(gold, k=num_distractors, pool=CAND_POOL)
    if not opts: return None, None
    scores = [_seq_logprob_with_hooks(prompt, o, fwd_hooks=[]) for o in opts]
    pred = opts[int(torch.tensor(scores).argmax().item())]
    return pred, gold

@torch.no_grad()
def compute_baseline_and_S(df_eval: pd.DataFrame, save_subset=True):
    hits = total = 0
    S_idx = []
    t0 = time.time()
    for i, (_, row) in enumerate(df_eval.iterrows(), 1):
        pred, gold = _eval_one_row_baseline(row)
        if pred is None: continue
        ok = int(pred == gold)
        hits += ok; total += 1
        if ok: S_idx.append(int(row.name))
        if total % LOG_EVERY == 0:
            acc = 100.0 * hits / total
            log(f"[BASELINE] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}% elapsed={time.time()-t0:0.1f}s")
    acc = 100.0 * hits / max(total,1)
    log(f"[RESULT] baseline acc@1={acc:0.2f}% (t={time.time()-t0:0.1f}s)")
    log(f"[BASELINE] |S| (aciertos) = {len(S_idx)} de {len(df_eval)}")
    if save_subset:
        try:
            with open(SUBSET_S_JSON, "w", encoding="utf-8") as f:
                json.dump(S_idx, f)
            # preview CSV
            df_eval.loc[S_idx, ["target_token"]].head(20).to_csv(SUBSET_S_CSV, index_label="row_idx")
        except Exception as e:
            log(f"[WARN] No se pudo guardar subset S: {e}")
    return acc, S_idx

# ---------- Evaluadores (se ejecutan SOLO sobre S) ----------
@torch.no_grad()
def eval_gold_only(df_eval, idx_S, layer, alpha, mode, hook_name):
    hits = total = 0; t0 = time.time()
    for c, i in enumerate(idx_S, 1):
        row = df_eval.loc[i]
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold_s = ensure_space_prefix(row["target_token"])
        opts   = [gold_s] + sample_distractors_safe(gold_s, k=NUM_DISTRACT, pool=CAND_POOL)

        surf_n  = gold_s.strip().lower()
        concept = SURF2CONCEPT.get(surf_n) or gold_s.strip()
        hyper   = SURF2HYPERNYM.get(surf_n)
        d_vec   = build_concept_dir_conceptual(concept, hyper, layer=layer, hook_name=hook_name)

        scores = []
        for o in opts:
            if o == gold_s:
                lp = seq_logprob_ablate_allpos(prompt, o, layer, hook_name, d_vec, alpha, mode)
            else:
                lp = _seq_logprob_with_hooks(prompt, o, fwd_hooks=[])
            scores.append(lp)

        pred  = opts[int(torch.tensor(scores).argmax().item())]
        hits += int(pred == gold_s); total += 1
        if total % LOG_EVERY == 0:
            acc = 100.0 * hits / total
            log(f"[GOLD_ONLY] {total:3d}/{len(idx_S):3d} acc@1={acc:5.2f}% elapsed={time.time()-t0:0.1f}s")
    return 100.0 * hits / max(total,1)

@torch.no_grad()
def eval_gold_dir_allpos(df_eval, idx_S, layer, alpha, mode, hook_name):
    hits = total = 0; t0 = time.time()
    for c, i in enumerate(idx_S, 1):
        row = df_eval.loc[i]
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold_s = ensure_space_prefix(row["target_token"])
        opts   = [gold_s] + sample_distractors_safe(gold_s, k=NUM_DISTRACT, pool=CAND_POOL)

        surf_n  = gold_s.strip().lower()
        concept = SURF2CONCEPT.get(surf_n) or gold_s.strip()
        hyper   = SURF2HYPERNYM.get(surf_n)
        d_vec   = build_concept_dir_conceptual(concept, hyper, layer=layer, hook_name=hook_name)

        scores = []
        for o in opts:
            lp = seq_logprob_ablate_allpos(prompt, o, layer, hook_name, d_vec, alpha, mode)
            scores.append(lp)

        pred  = opts[int(torch.tensor(scores).argmax().item())]
        hits += int(pred == gold_s); total += 1
        if total % LOG_EVERY == 0:
            acc = 100.0 * hits / total
            log(f"[GOLD_ALL ] {total:3d}/{len(idx_S):3d} acc@1={acc:5.2f}% elapsed={time.time()-t0:0.1f}s")
    return 100.0 * hits / max(total,1)

# ---------- Reanudar: evitar recomputar combos ya guardados ----------
def _load_done_rows(csv_path: str):
    if not os.path.exists(csv_path): return set()
    try:
        df_done = pd.read_csv(csv_path)
        keys = set(
            (str(r["form"]), str(r["hook"]), str(r["mode"]), str(r["loc"]), int(r["layer"]), float(r["alpha"]))
            for _, r in df_done.iterrows()
            if str(r["form"]) != "baseline"
        )
        return keys
    except Exception:
        return set()

def _append_result_row(row: dict):
    exists = os.path.exists(RESULTS_CSV)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode=("a" if exists else "w"),
                               header=not exists, index=False)

# ---------- ORQUESTADOR ----------
def run_method_grid(df_all: pd.DataFrame):
    log("==== INICIO (baseline + subset S) ====")
    base_acc, idx_S = compute_baseline_and_S(df_all, save_subset=True)

    # Baseline en CSV (una sola vez)
    if not os.path.exists(RESULTS_CSV):
        _append_result_row({"form": "baseline", "hook": "-", "mode": "-", "loc": "-",
                            "layer": -1, "alpha": -1, "acc@1": base_acc, "n_items": len(df_all)})

    # Preparar “done set” para reanudar
    done = _load_done_rows(RESULTS_CSV)
    log(f"[RESUME] combos ya hechos: {len(done)}")

    # Grilla sobre S
    forms = ["gold_only", "gold_dir_allpos"]
    total_combos = len(forms)*len(HOOKS)*len(MODES)*len(LAYERS_BY_LOC)*len(ALPHAS)
    k = 0
    for hook_name in HOOKS:
        for mode in MODES:
            for loc, L in LAYERS_BY_LOC.items():
                for alpha in ALPHAS:
                    for form in forms:
                        k += 1
                        key = (form, hook_name, mode, loc, L, float(alpha))
                        if key in done:
                            log(f"[SKIP {k}/{total_combos}] {key}")
                            continue

                        log(f"\n-- FORM: {form} --")
                        log(f"[GRID {k}/{total_combos}] loc={loc} L={L:02d} hook={hook_name} mode={mode} α={alpha}")
                        t1 = time.time()
                        if form == "gold_only":
                            acc = eval_gold_only(df_all, idx_S, layer=L, alpha=alpha, mode=mode, hook_name=hook_name)
                        elif form == "gold_dir_allpos":
                            acc = eval_gold_dir_allpos(df_all, idx_S, layer=L, alpha=alpha, mode=mode, hook_name=hook_name)
                        else:
                            raise ValueError(form)

                        row = {"form": form, "hook": hook_name, "mode": mode, "loc": loc,
                               "layer": L, "alpha": alpha, "acc@1": acc, "n_items": len(idx_S)}
                        _append_result_row(row)
                        log(f"[RESULT] {form:12s} acc@1={acc:0.2f}%  (t={time.time()-t1:0.1f}s)")

    log("\n==== FIN GRID ====")
    try:
        df_out = pd.read_csv(RESULTS_CSV)
        print(df_out.sort_values(["form","hook","mode","loc","layer","alpha"]).to_string(index=False))
    except Exception:
        pass

# ===================== EJECUTAR =====================
run_method_grid(df)


[23:48:59] ==== INICIO (baseline + subset S) ====
[23:49:02] [BASELINE]  20/322 acc@1=80.00% elapsed=2.7s
[23:49:04] [BASELINE]  40/322 acc@1=72.50% elapsed=5.1s
[23:49:07] [BASELINE]  60/322 acc@1=75.00% elapsed=7.6s
[23:49:09] [BASELINE]  80/322 acc@1=71.25% elapsed=10.0s
[23:49:12] [BASELINE] 100/322 acc@1=66.00% elapsed=12.4s
[23:49:14] [BASELINE] 120/322 acc@1=63.33% elapsed=14.8s
[23:49:16] [BASELINE] 140/322 acc@1=60.71% elapsed=17.2s
[23:49:19] [BASELINE] 160/322 acc@1=61.25% elapsed=19.7s
[23:49:21] [BASELINE] 180/322 acc@1=61.11% elapsed=22.1s
[23:49:24] [BASELINE] 200/322 acc@1=59.00% elapsed=24.5s
[23:49:26] [BASELINE] 220/322 acc@1=58.18% elapsed=26.9s
[23:49:28] [BASELINE] 240/322 acc@1=57.92% elapsed=29.4s
[23:49:31] [BASELINE] 260/322 acc@1=56.92% elapsed=31.8s
[23:49:33] [BASELINE] 280/322 acc@1=56.79% elapsed=34.2s
[23:49:36] [BASELINE] 300/322 acc@1=54.33% elapsed=36.6s
[23:49:38] [BASELINE] 320/322 acc@1=53.75% elapsed=39.1s
[23:49:38] [RESULT] baseline acc@1=53.42%

In [9]:
# ===================== LOBOTOMÍA – DOE METODOLÓGICO =====================
# Requiere: model (TransformerLens HookedTransformer) y df con columnas:
#   - target_token (str), prompt_cloze o prompt_effective (str)
#   - concept, hypernym (opcionales)
# Salidas: CSV incremental + subset S de baseline

import os, json, time, math, random
from typing import List, Tuple, Optional, Dict
import pandas as pd
import torch
import torch.nn.functional as F

# ---------- Config principal (metodología) ----------
SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

# Intensidades (metodología): 1.5 (baja), 3.5 (media), 6.0 (alta)
ALPHAS = [1.5, 3.5, 6.0]   # (H1C: dosis-respuesta)

# Localizaciones (metodología): capa media (50%) y tardía (75%)
def _layer_index(frac: float) -> int:
    nL = int(getattr(model.cfg, "n_layers", 12))
    # clamp y redondeo a entero de 0..nL-1
    return max(0, min(nL-1, int(round((nL-1) * frac))))

LAYERS_BY_LOC = {
    "mid":  _layer_index(0.25)
}

# Técnica de ablación (factor A): sustracción vs proyección
# - add     => a' = a0 - α d          (sustracción)
# - project => a' = a0 - α·proj_d(a0)  (escalamos la proyección por α; α=1 es borrado “perfecto”)
MODES = ["add", "project"]

# Tipos de hook (factor adicional). Metodológicamente basta resid_pre; los otros son exploratorios.
HOOKS = ["hook_resid_pre"]            # puedes ampliar: ["hook_resid_pre","hook_mlp_out","hook_attn_out"]

# Distractores y logging
NUM_DISTRACT   = 3
LOG_EVERY      = 20

# Archivos de salida (checkpoint/reanudable)
RESULTS_CSV    = "grid_results_method.csv"
SUBSET_S_JSON  = "subset_S_indices.json"
SUBSET_S_CSV   = "subset_S_preview.csv"

# ---------- Utilidades base ----------
def log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

try:
    DEVICE
except NameError:
    try:
        DEVICE = next(model.parameters()).device
    except Exception:
        DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def ensure_space_prefix(s: str) -> str:
    s = str(s or "")
    return s if (len(s) > 0 and s[0].isspace()) else (" " + s)

# Construcción de mapas surface→concept/hypernym y pool de distractores
GENERIC_HS = {"thing","object","device","device type","stuff","material","entity",
              "concept","type","kind","kind of","being","animal","person"}

def build_surface_maps(df_in: pd.DataFrame):
    d = df_in.copy()
    d["surface"]   = d["target_token"].astype(str).str.strip()
    d["surface_n"] = d["surface"].str.lower()

    d["concept"]  = d.get("concept", pd.Series([None]*len(d))).astype(str).str.strip()
    d["hypernym"] = d.get("hypernym", pd.Series([None]*len(d))).astype(str).str.strip().str.lower()

    surf_grp = d.groupby("surface_n")
    surf2concept  = (surf_grp["concept"]
                     .agg(lambda s: s.value_counts().idxmax() if len(s.dropna()) else None)
                     .to_dict())
    surf2hypernym = (surf_grp["hypernym"]
                     .agg(lambda s: s.value_counts().idxmax() if len(s.dropna()) else None)
                     .to_dict())

    for k, h in list(surf2hypernym.items()):
        if (h is None) or (h in GENERIC_HS):
            surf2hypernym[k] = None

    cand_pool = sorted({
        ensure_space_prefix(s)
        for s in d["surface"].unique().tolist()
        if isinstance(s, str) and s.strip()
    })
    return surf2concept, surf2hypernym, cand_pool

SURF2CONCEPT, SURF2HYPERNYM, CAND_POOL = build_surface_maps(df)

def sample_distractors_safe(gold_surface: str, k: int, pool: List[str]):
    gold_norm = str(gold_surface or "").strip().lower()
    cand = [w for w in pool if str(w).strip().lower() != gold_norm]
    if len(cand) <= k:
        return cand
    rnd = random.Random(SEED + hash(gold_norm) % (10**6))
    return rnd.sample(cand, k)

# ---------- Tokenización/activaciones ----------
def to_tok_ids(text: str) -> torch.Tensor:
    return model.to_tokens(text, prepend_bos=False).to(DEVICE)[0]

def find_subseq(haystack_ids: torch.Tensor, needle_ids: torch.Tensor):
    H, N = len(haystack_ids), len(needle_ids)
    if N == 0 or N > H: return None
    for i in range(H - N + 1):
        if torch.equal(haystack_ids[i:i+N], needle_ids):
            return (i, i+N)
    return None

def get_act_for_positions(prompt_text: str, layer: int, hook_name: str, pos_indices):
    toks = model.to_tokens(prompt_text, prepend_bos=False).to(DEVICE)
    _, cache = model.run_with_cache(toks, remove_batch_dim=False)
    H = cache[f'blocks.{layer}.{hook_name}']  # [1, seq, d_model]
    idx = torch.as_tensor(list(pos_indices), device=H.device)
    return H[0, idx, :].detach()

# ---------- Dirección “conceptual” (concepto + hiperónimo opcional) ----------
_CONCEPT_CACHE: Dict = {}

@torch.no_grad()
def build_concept_dir_conceptual(concept: str,
                                 hypernym: Optional[str],
                                 layer: int,
                                 hook_name: str = "hook_resid_pre",
                                 base_templates = (
                                     "A {X} is a",
                                     "The {X} is commonly used",
                                     "People often use the {X}",
                                     "This {X} is useful",
                                 )) -> torch.Tensor:
    key = (layer, hook_name, (concept or "").lower().strip(), (hypernym or None))
    if key in _CONCEPT_CACHE:
        return _CONCEPT_CACHE[key]

    X = (concept or "").strip()
    if not X:
        raise ValueError("concept vacío en build_concept_dir_conceptual")

    templates = list(base_templates)
    if hypernym and hypernym not in GENERIC_HS:
        templates.append(f"A {{X}} is a {hypernym}")

    x_ids = to_tok_ids(" " + X)
    acts = []
    for tpl in templates:
        prompt = tpl.replace("{X}", X)
        p_ids  = to_tok_ids(prompt)
        match  = find_subseq(p_ids, x_ids)
        if match is None:
            start, end = len(p_ids)-1, len(p_ids)
        else:
            start, end = match
        pos = list(range(start, end))
        a = get_act_for_positions(prompt, layer, hook_name, pos)  # [m, d]
        acts.append(a.mean(0))
    d = torch.stack(acts, dim=0).mean(0)
    d = d / (d.norm() + 1e-8)
    _CONCEPT_CACHE[key] = d
    return d

# ---------- Hooks de ablación ----------
def make_ablate_hook(direction: torch.Tensor, alpha: float, mode: str, positions):
    v_hat = direction / (direction.norm() + 1e-8)
    pos_idx = torch.tensor(sorted({int(i) for i in positions}))

    def hook_fn(h, hook):
        v = v_hat.to(h.device)
        idx = pos_idx.to(h.device)
        h_out = h.clone()
        sl = h_out[:, idx, :]
        if mode == "add":  # sustracción
            sl = sl - alpha * v
        elif mode == "project":  # proyección (posible sobre/under con α)
            proj = (sl @ v)[..., None] * v
            sl = sl - alpha * proj
        else:
            raise ValueError("mode debe ser 'add' o 'project'")
        h_out[:, idx, :] = sl
        return h_out
    return hook_fn

# ---------- LogProb genérico con hooks ----------
@torch.no_grad()
def _seq_logprob_with_hooks(prompt: str, continuation: str, fwd_hooks=None):
    if fwd_hooks is None: fwd_hooks = []
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(DEVICE)
    P = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids = model.to_tokens(continuation, prepend_bos=False).to(DEVICE)[0]

    logits = model.run_with_hooks(toks_all, fwd_hooks=fwd_hooks, return_type="logits")
    logprobs = F.log_softmax(logits, dim=-1)
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

# Ablar SOLO los subtokens de la opción
@torch.no_grad()
def seq_logprob_ablate_option(prompt: str, option: str,
                              layer: int, hook_name: str,
                              direction: torch.Tensor,
                              alpha: float, mode: str):
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    option = ensure_space_prefix(option)

    toks_all  = model.to_tokens(prompt + option, prepend_bos=False).to(DEVICE)
    P         = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids  = model.to_tokens(option, prepend_bos=False).to(DEVICE)[0]

    match = find_subseq(toks_all[0, P:], cont_ids)
    if match is None:
        opt_pos = [toks_all.shape[1]-1]
    else:
        a, b   = match
        opt_pos= list(range(P + a, P + b))

    steer = make_ablate_hook(direction, alpha, mode, opt_pos)
    logits = model.run_with_hooks(toks_all, fwd_hooks=[(f'blocks.{layer}.{hook_name}', steer)],
                                  return_type="logits")
    logprobs = F.log_softmax(logits, dim=-1)
    pref = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

# Ablar TODA la continuación
@torch.no_grad()
def seq_logprob_ablate_allpos(prompt: str, option: str,
                              layer: int, hook_name: str,
                              direction: torch.Tensor,
                              alpha: float, mode: str):
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    option = ensure_space_prefix(option)

    toks_all  = model.to_tokens(prompt + option, prepend_bos=False).to(DEVICE)
    P         = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids  = model.to_tokens(option, prepend_bos=False).to(DEVICE)[0]

    opt_pos   = list(range(P, toks_all.shape[1]))
    steer     = make_ablate_hook(direction, alpha, mode, opt_pos)

    logits    = model.run_with_hooks(toks_all, fwd_hooks=[(f'blocks.{layer}.{hook_name}', steer)],
                                     return_type="logits")
    logprobs  = F.log_softmax(logits, dim=-1)
    pref      = logprobs[0, P-1:-1]
    return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

# ---------- Baseline y subset S ----------
@torch.no_grad()
def _eval_one_row_baseline(row, num_distractors=NUM_DISTRACT):
    prompt = row.get("prompt_cloze") or row.get("prompt_effective")
    gold   = ensure_space_prefix(row["target_token"])
    opts   = [gold] + sample_distractors_safe(gold, k=num_distractors, pool=CAND_POOL)
    if not opts: return None, None
    scores = [_seq_logprob_with_hooks(prompt, o, fwd_hooks=[]) for o in opts]
    pred = opts[int(torch.tensor(scores).argmax().item())]
    return pred, gold

@torch.no_grad()
def compute_baseline_and_S(df_eval: pd.DataFrame, save_subset=True):
    hits = total = 0
    S_idx = []
    t0 = time.time()
    for i, (_, row) in enumerate(df_eval.iterrows(), 1):
        pred, gold = _eval_one_row_baseline(row)
        if pred is None: continue
        ok = int(pred == gold)
        hits += ok; total += 1
        if ok: S_idx.append(int(row.name))
        if total % LOG_EVERY == 0:
            acc = 100.0 * hits / total
            log(f"[BASELINE] {total:3d}/{len(df_eval):3d} acc@1={acc:5.2f}% elapsed={time.time()-t0:0.1f}s")
    acc = 100.0 * hits / max(total,1)
    log(f"[RESULT] baseline acc@1={acc:0.2f}% (t={time.time()-t0:0.1f}s)")
    log(f"[BASELINE] |S| (aciertos) = {len(S_idx)} de {len(df_eval)}")
    if save_subset:
        try:
            with open(SUBSET_S_JSON, "w", encoding="utf-8") as f:
                json.dump(S_idx, f)
            # preview CSV
            df_eval.loc[S_idx, ["target_token"]].head(20).to_csv(SUBSET_S_CSV, index_label="row_idx")
        except Exception as e:
            log(f"[WARN] No se pudo guardar subset S: {e}")
    return acc, S_idx

# ---------- Evaluadores (se ejecutan SOLO sobre S) ----------
@torch.no_grad()
def eval_gold_only(df_eval, idx_S, layer, alpha, mode, hook_name):
    hits = total = 0; t0 = time.time()
    for c, i in enumerate(idx_S, 1):
        row = df_eval.loc[i]
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold_s = ensure_space_prefix(row["target_token"])
        opts   = [gold_s] + sample_distractors_safe(gold_s, k=NUM_DISTRACT, pool=CAND_POOL)

        surf_n  = gold_s.strip().lower()
        concept = SURF2CONCEPT.get(surf_n) or gold_s.strip()
        hyper   = SURF2HYPERNYM.get(surf_n)
        d_vec   = build_concept_dir_conceptual(concept, hyper, layer=layer, hook_name=hook_name)

        scores = []
        for o in opts:
            if o == gold_s:
                lp = seq_logprob_ablate_allpos(prompt, o, layer, hook_name, d_vec, alpha, mode)
            else:
                lp = _seq_logprob_with_hooks(prompt, o, fwd_hooks=[])
            scores.append(lp)

        pred  = opts[int(torch.tensor(scores).argmax().item())]
        hits += int(pred == gold_s); total += 1
        if total % LOG_EVERY == 0:
            acc = 100.0 * hits / total
            log(f"[GOLD_ONLY] {total:3d}/{len(idx_S):3d} acc@1={acc:5.2f}% elapsed={time.time()-t0:0.1f}s")
    return 100.0 * hits / max(total,1)

@torch.no_grad()
def eval_gold_dir_allpos(df_eval, idx_S, layer, alpha, mode, hook_name):
    hits = total = 0; t0 = time.time()
    for c, i in enumerate(idx_S, 1):
        row = df_eval.loc[i]
        prompt = row.get("prompt_cloze") or row.get("prompt_effective")
        gold_s = ensure_space_prefix(row["target_token"])
        opts   = [gold_s] + sample_distractors_safe(gold_s, k=NUM_DISTRACT, pool=CAND_POOL)

        surf_n  = gold_s.strip().lower()
        concept = SURF2CONCEPT.get(surf_n) or gold_s.strip()
        hyper   = SURF2HYPERNYM.get(surf_n)
        d_vec   = build_concept_dir_conceptual(concept, hyper, layer=layer, hook_name=hook_name)

        scores = []
        for o in opts:
            lp = seq_logprob_ablate_allpos(prompt, o, layer, hook_name, d_vec, alpha, mode)
            scores.append(lp)

        pred  = opts[int(torch.tensor(scores).argmax().item())]
        hits += int(pred == gold_s); total += 1
        if total % LOG_EVERY == 0:
            acc = 100.0 * hits / total
            log(f"[GOLD_ALL ] {total:3d}/{len(idx_S):3d} acc@1={acc:5.2f}% elapsed={time.time()-t0:0.1f}s")
    return 100.0 * hits / max(total,1)

# ---------- Reanudar: evitar recomputar combos ya guardados ----------
def _load_done_rows(csv_path: str):
    if not os.path.exists(csv_path): return set()
    try:
        df_done = pd.read_csv(csv_path)
        keys = set(
            (str(r["form"]), str(r["hook"]), str(r["mode"]), str(r["loc"]), int(r["layer"]), float(r["alpha"]))
            for _, r in df_done.iterrows()
            if str(r["form"]) != "baseline"
        )
        return keys
    except Exception:
        return set()

def _append_result_row(row: dict):
    exists = os.path.exists(RESULTS_CSV)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode=("a" if exists else "w"),
                               header=not exists, index=False)

# ---------- ORQUESTADOR ----------
def run_method_grid(df_all: pd.DataFrame):
    log("==== INICIO (baseline + subset S) ====")
    base_acc, idx_S = compute_baseline_and_S(df_all, save_subset=True)

    # Preparar “done set” para reanudar
    done = _load_done_rows(RESULTS_CSV)
    log(f"[RESUME] combos ya hechos: {len(done)}")

    # Grilla sobre S
    forms = ["gold_only", "gold_dir_allpos"]
    total_combos = len(forms)*len(HOOKS)*len(MODES)*len(LAYERS_BY_LOC)*len(ALPHAS)
    k = 0
    for hook_name in HOOKS:
        for mode in MODES:
            for loc, L in LAYERS_BY_LOC.items():
                for alpha in ALPHAS:
                    for form in forms:
                        k += 1
                        key = (form, hook_name, mode, loc, L, float(alpha))
                        if key in done:
                            log(f"[SKIP {k}/{total_combos}] {key}")
                            continue

                        log(f"\n-- FORM: {form} --")
                        log(f"[GRID {k}/{total_combos}] loc={loc} L={L:02d} hook={hook_name} mode={mode} α={alpha}")
                        t1 = time.time()
                        if form == "gold_only":
                            acc = eval_gold_only(df_all, idx_S, layer=L, alpha=alpha, mode=mode, hook_name=hook_name)
                        elif form == "gold_dir_allpos":
                            acc = eval_gold_dir_allpos(df_all, idx_S, layer=L, alpha=alpha, mode=mode, hook_name=hook_name)
                        else:
                            raise ValueError(form)

                        row = {"form": form, "hook": hook_name, "mode": mode, "loc": loc,
                               "layer": L, "alpha": alpha, "acc@1": acc, "n_items": len(idx_S)}
                        _append_result_row(row)
                        log(f"[RESULT] {form:12s} acc@1={acc:0.2f}%  (t={time.time()-t1:0.1f}s)")

    log("\n==== FIN GRID ====")
    try:
        df_out = pd.read_csv(RESULTS_CSV)
        print(df_out.sort_values(["form","hook","mode","loc","layer","alpha"]).to_string(index=False))
    except Exception:
        pass

# ===================== EJECUTAR =====================
run_method_grid(df)


[00:04:36] ==== INICIO (baseline + subset S) ====
[00:04:38] [BASELINE]  20/322 acc@1=80.00% elapsed=2.5s
[00:04:41] [BASELINE]  40/322 acc@1=72.50% elapsed=4.9s
[00:04:43] [BASELINE]  60/322 acc@1=75.00% elapsed=7.3s
[00:04:46] [BASELINE]  80/322 acc@1=71.25% elapsed=9.8s
[00:04:48] [BASELINE] 100/322 acc@1=66.00% elapsed=12.2s
[00:04:51] [BASELINE] 120/322 acc@1=63.33% elapsed=14.6s
[00:04:53] [BASELINE] 140/322 acc@1=60.71% elapsed=17.1s
[00:04:55] [BASELINE] 160/322 acc@1=61.25% elapsed=19.5s
[00:04:58] [BASELINE] 180/322 acc@1=61.11% elapsed=22.0s
[00:05:00] [BASELINE] 200/322 acc@1=59.00% elapsed=24.5s
[00:05:03] [BASELINE] 220/322 acc@1=58.18% elapsed=26.9s
[00:05:05] [BASELINE] 240/322 acc@1=57.92% elapsed=29.3s
[00:05:08] [BASELINE] 260/322 acc@1=56.92% elapsed=31.8s
[00:05:10] [BASELINE] 280/322 acc@1=56.79% elapsed=34.3s
[00:05:13] [BASELINE] 300/322 acc@1=54.33% elapsed=36.7s
[00:05:15] [BASELINE] 320/322 acc@1=53.75% elapsed=39.2s
[00:05:15] [RESULT] baseline acc@1=53.42% 

# hacer una prueba con el mayor detalle posible 
#   HACER DIAGRAMAS 
# EJERCICIO UN UNICO INTENTO PERO A DETALLLE DESDE PUNTO MATEMATICO Y ESTADISTICO
#  ENTENDER LA FUBCION DE LA LIBRERIA


In [12]:
# ===================== PoC CAV + Hook (modelo ya cargado, en GPU/FP16) =====================
# Requiere: `model` (HookedTransformer) YA cargado.
# Usa L12 + hook_resid_pre para construir CAV del concepto "electrical component".

import os, math, random, time
from typing import List, Dict
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.svm import LinearSVC

# ------------------ Config ------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LAYER_FOR_CAV = 12                # capa objetivo (L12)
HOOK_NAME     = "hook_resid_pre"  # punto de extracción e intervención
CAV_PATH      = "cav_electrical_component.pt"

# Factores experimentales
FACTOR_A_MODE  = "project"        # "add" | "project"
FACTOR_C_ALPHA = 1.0              # intensidad

# Ítem de prueba
PROMPT_TEST = "A fuse is an"
CANDIDATES  = ["electrical component", "animal", "vehicle", "furniture"]

# ——— Asegurar GPU + dtype destino (usa tus constantes si las tienes) ———
DTYPE_TARGET = torch.float16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Si el modelo no quedó en GPU al cargarlo, lo movemos ahora
try:
    _param = next(model.parameters())
    if _param.device.type != "cuda":
        try:
            model = model.to(DEVICE)
        except Exception:
            pass
except StopIteration:
    pass

# Refrescar device/dtype "reales" desde el modelo
MODEL_DEVICE = next(model.parameters()).device
MODEL_DTYPE  = next(model.parameters()).dtype

# Si quieres forzar FP16 en el modelo ya cargado (si fue FP32):
if MODEL_DTYPE != DTYPE_TARGET and MODEL_DEVICE.type == "cuda":
    try:
        model = model.to(dtype=DTYPE_TARGET)
        MODEL_DTYPE = DTYPE_TARGET
    except Exception:
        # Si tu build/layernorms no soportan cast completo, lo ignoramos sin romper.
        pass

print(f"[INFO] model.device = {MODEL_DEVICE}, model.dtype = {MODEL_DTYPE}")

[INFO] model.device = cuda:0, model.dtype = torch.float16


In [13]:

# ------------------ Utilidades ------------------
def ensure_space_prefix(s: str) -> str:
    s = str(s or "")
    return s if (len(s) > 0 and s[0].isspace()) else (" " + s)

def _to_dev(x: torch.Tensor) -> torch.Tensor:
    return x.to(MODEL_DEVICE)

def to_tokens_ids(text: str) -> torch.Tensor:
    # sin BOS; coherente en todo el script
    return model.to_tokens(text, prepend_bos=False).to(MODEL_DEVICE)[0]

@torch.no_grad()
def get_lastpos_resid_pre(prompt: str, layer: int) -> torch.Tensor:
    # Activación residual pre-capa en el último token del prompt
    toks = model.to_tokens(prompt, prepend_bos=False).to(MODEL_DEVICE)
    # autocast para FP16 estable (solo en GPU)
    use_amp = (MODEL_DEVICE.type == "cuda")
    cm = torch.cuda.amp.autocast(enabled=use_amp, dtype=MODEL_DTYPE if use_amp else None)
    with cm:
        _, cache = model.run_with_cache(
            toks, remove_batch_dim=False, names_filter=f"blocks.{layer}.{HOOK_NAME}"
        )
    h = cache[f"blocks.{layer}.{HOOK_NAME}"]  # [1, seq, d_model]
    return h[0, -1, :].detach()               # vector (d_model,)

def make_steer_hook(cav: torch.Tensor, alpha: float, mode: str):
    # cav ya debe estar en el device/dtype del modelo
    u = cav / (torch.norm(cav) + 1e-12)
    def _hook(h, hook):
        # h: [batch, seq, d_model]
        if mode == "add":
            return h - alpha * u
        elif mode == "project":
            proj = torch.einsum("bsd,d->bs", h, u).unsqueeze(-1) * u
            return h - proj
        else:
            return h
    return _hook

@torch.no_grad()
def continuation_logprob(prompt: str, continuation: str) -> float:
    prompt = prompt.rstrip()
    if not prompt.endswith(" "): prompt += " "
    continuation = ensure_space_prefix(continuation)

    toks_all = model.to_tokens(prompt + continuation, prepend_bos=False).to(MODEL_DEVICE)
    P        = model.to_tokens(prompt, prepend_bos=False).shape[1]
    cont_ids = model.to_tokens(continuation, prepend_bos=False).to(MODEL_DEVICE)[0]

    use_amp = (MODEL_DEVICE.type == "cuda")
    cm = torch.cuda.amp.autocast(enabled=use_amp, dtype=MODEL_DTYPE if use_amp else None)
    with cm:
        logits = model(toks_all)
        logp   = F.log_softmax(logits, dim=-1)[0]
        pref   = logp[P-1:-1]
        return pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()

def softmax_local_from_logps(logps: Dict[str, float]) -> Dict[str, float]:
    vs = np.array(list(logps.values()), dtype=np.float64)
    vs = vs - vs.max()
    ps = np.exp(vs); ps = ps / ps.sum()
    return {k: float(p) for k, p in zip(logps.keys(), ps)}

@torch.no_grad()
def score_set(prompt: str, opts: List[str], fwd_hooks=None) -> Dict[str, float]:
    fwd_hooks = fwd_hooks or []
    logps = {}
    use_amp = (MODEL_DEVICE.type == "cuda")
    for o in opts:
        prompt_ = prompt.rstrip()
        if not prompt_.endswith(" "): prompt_ += " "
        cont_ = ensure_space_prefix(o)

        toks_all = model.to_tokens(prompt_ + cont_, prepend_bos=False).to(MODEL_DEVICE)
        P        = model.to_tokens(prompt_, prepend_bos=False).shape[1]
        cont_ids = model.to_tokens(cont_, prepend_bos=False).to(MODEL_DEVICE)[0]

        cm = torch.cuda.amp.autocast(enabled=use_amp, dtype=MODEL_DTYPE if use_amp else None)
        with cm:
            if fwd_hooks:
                logits = model.run_with_hooks(toks_all, fwd_hooks=fwd_hooks, return_type="logits")
            else:
                logits = model(toks_all)
            logp = F.log_softmax(logits, dim=-1)[0]
            pref = logp[P-1:-1]
            lp   = pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()
        logps[o] = lp
    return softmax_local_from_logps(logps)

# ------------------ 20 Positivos y 20 Negativos ------------------
positives = [
    "A battery is an electrical component",
    "A resistor is an electrical component",
    "A capacitor is an electrical component",
    "A diode is an electrical component",
    "A transistor is an electrical component",
    "An inductor is an electrical component",
    "A fuse is an electrical component",
    "A relay is an electrical component",
    "A transformer is an electrical component",
    "A rectifier is an electrical component",
    "A varistor is an electrical component",
    "A potentiometer is an electrical component",
    "An LED is an electrical component",
    "A switch is an electrical component",
    "A photodiode is an electrical component",
    "A thermistor is an electrical component",
    "A buzzer is an electrical component",
    "A microphone is an electrical component",
    "A speaker is an electrical component",
    "A socket is an electrical component",
]
negatives = [
    "A tree is a living organism",
    "A river is a natural waterway",
    "A mountain is a landform",
    "A poet is a person who writes poems",
    "A cat is a domesticated animal",
    "A city is a large settlement",
    "A bicycle is a human-powered vehicle",
    "A chair is a piece of furniture",
    "A painting is a work of art",
    "A sandwich is a type of food",
    "A concert is a musical performance",
    "A novel is a long narrative",
    "A beach is a sandy shore",
    "A shoe is an item of clothing",
    "A window is an opening in a wall",
    "A classroom is a learning space",
    "A sunrise is a daily event",
    "A movie is a form of entertainment",
    "A keyboard is a computer peripheral",
    "A museum is a cultural institution",
]

# ------------------ 1) Extraer activaciones y entrenar SVM (CAV) ------------------
X, y = [], []

with torch.no_grad():
    for s in positives:
        X.append(get_lastpos_resid_pre(s, LAYER_FOR_CAV).float().cpu().numpy()); y.append(1)
    for s in negatives:
        X.append(get_lastpos_resid_pre(s, LAYER_FOR_CAV).float().cpu().numpy()); y.append(0)

X = np.stack(X, axis=0)           # (40, d_model)
y = np.array(y, dtype=np.int32)   # (40,)

# SVM se entrena en CPU (OK). Si quieres, puedes usar SGDClassifier también.
clf = LinearSVC(C=1.0, random_state=SEED, dual=False, max_iter=5000)
clf.fit(X, y)

w = clf.coef_.reshape(-1)         # vector normal del hiperplano
w = w / (np.linalg.norm(w) + 1e-12)
cav = torch.tensor(w, device=MODEL_DEVICE, dtype=MODEL_DTYPE)

torch.save(cav, CAV_PATH)
print(f"[OK] CAV guardado en {CAV_PATH} — dim={cav.numel()} — device={cav.device} dtype={cav.dtype}")

# ------------------ 2) Baseline vs Intervención ------------------
print("\n=== BASELINE ===")
probs_base = score_set(PROMPT_TEST, CANDIDATES)
print("Prompt:", PROMPT_TEST)
for k, v in probs_base.items():
    print(f"  P({k}) = {v:.4f}")

print("\n=== INTERVENCIÓN ===")
hook = make_steer_hook(cav, alpha=FACTOR_C_ALPHA, mode=FACTOR_A_MODE)
probs_int = score_set(
    PROMPT_TEST, CANDIDATES,
    fwd_hooks=[(f"blocks.{LAYER_FOR_CAV}.{HOOK_NAME}", hook)]
)
for k, v in probs_int.items():
    print(f"  P({k}) = {v:.4f}")

print("\n=== DELTAS (Interv - Base) ===")
for k in CANDIDATES:
    d = probs_int[k] - probs_base[k]
    print(f"  Δ {k:>20s} = {d:+.4f}")

if MODEL_DEVICE.type == "cuda":
    torch.cuda.synchronize()
print("\nNota: si la ablación funciona, P('electrical component') debería caer (Δ < 0).")

/tmp/ipykernel_12417/1258798686.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  cm = torch.cuda.amp.autocast(enabled=use_amp, dtype=MODEL_DTYPE if use_amp else None)


[OK] CAV guardado en cav_electrical_component.pt — dim=1024 — device=cuda:0 dtype=torch.float16

=== BASELINE ===
Prompt: A fuse is an
  P(electrical component) = 0.7798
  P(animal) = 0.0712
  P(vehicle) = 0.1225
  P(furniture) = 0.0265

=== INTERVENCIÓN ===


/tmp/ipykernel_12417/1258798686.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  cm = torch.cuda.amp.autocast(enabled=use_amp, dtype=MODEL_DTYPE if use_amp else None)


  P(electrical component) = 0.9929
  P(animal) = 0.0034
  P(vehicle) = 0.0029
  P(furniture) = 0.0008

=== DELTAS (Interv - Base) ===
  Δ electrical component = +0.2131
  Δ               animal = -0.0679
  Δ              vehicle = -0.1196
  Δ            furniture = -0.0256

Nota: si la ablación funciona, P('electrical component') debería caer (Δ < 0).


In [14]:
# === Cargar e inspeccionar el CAV guardado ===
import torch, numpy as np
from sklearn.metrics import roc_auc_score

CAV_PATH = "cav_electrical_component.pt"
cav = torch.load(CAV_PATH, map_location=next(model.parameters()).device)

print("[CAV] shape:", tuple(cav.shape), "device:", cav.device, "dtype:", cav.dtype)
print("[CAV] ||cav||_2:", float(torch.norm(cav).item()))
print("[CAV] primeras 16 componentes:", cav[:16].detach().float().cpu().numpy())
print("[CAV] top-10 |componentes| (idx, valor):")
abs_c = torch.abs(cav).detach().float().cpu().numpy()
top_idx = abs_c.argsort()[-10:][::-1]
for i in top_idx:
    print(f"  {i:4d}  {float(cav[i]):+.6f}")


[CAV] shape: (1024,) device: cuda:0 dtype: torch.float16
[CAV] ||cav||_2: 1.0
[CAV] primeras 16 componentes: [-0.04107666 -0.00366783  0.03268433 -0.01062012  0.01856995 -0.04440308
 -0.02587891 -0.02815247 -0.05526733 -0.00191212 -0.03616333 -0.05267334
 -0.02157593  0.04537964 -0.05548096  0.00855255]
[CAV] top-10 |componentes| (idx, valor):
   268  +0.146973
   580  +0.130737
   990  -0.105103
   775  -0.096497
   624  +0.096252
   820  -0.095032
   946  -0.092529
   346  -0.089539
   356  +0.086853
    36  +0.079163


In [16]:
import torch, numpy as np
from sklearn.metrics import roc_auc_score

LAYER_FOR_CAV = 12
HOOK_NAME     = "hook_resid_pre"
CAV_PATH      = "cav_electrical_component.pt"

# Cargar CAV y pasarlo a float32 SOLO para análisis
cav = torch.load(CAV_PATH, map_location=next(model.parameters()).device)
cav_f32 = cav.detach().float()  # <- clave

print("[CAV] shape:", tuple(cav.shape), "device:", cav.device, "dtype:", cav.dtype)
print("[CAV(f32)] ||cav||_2:", float(torch.norm(cav_f32)))
print("[CAV(f32)] primeras 16:", cav_f32[:16].cpu().numpy())

# Utilidades
@torch.no_grad()
def resid_pre_last(model, text, layer, hook="hook_resid_pre"):
    toks = model.to_tokens(text, prepend_bos=False).to(next(model.parameters()).device)
    _, cache = model.run_with_cache(toks, remove_batch_dim=False,
                                    names_filter=f"blocks.{layer}.{hook}")
    # devolvemos SIEMPRE float32 para métricas
    return cache[f"blocks.{layer}.{hook}"][0, -1, :].detach().float()

def proj_scores(texts):
    scores = []
    for s in texts:
        h = resid_pre_last(model, s, layer=LAYER_FOR_CAV)
        # h: float32 ; cav_f32: float32
        scores.append(float(torch.dot(h, cav_f32)))
    return np.array(scores, dtype=np.float64)

# Usa tus listas
pos_scores = proj_scores(positives)
neg_scores = proj_scores(negatives)

print("\n[CHECK] Proyecciones sobre CAV (fp32)")
print("  pos mean/std/min/max:", pos_scores.mean(), pos_scores.std(), pos_scores.min(), pos_scores.max())
print("  neg mean/std/min/max:", neg_scores.mean(), neg_scores.std(), neg_scores.min(), neg_scores.max())
labels = np.array([1]*len(pos_scores) + [0]*len(neg_scores))
auc = roc_auc_score(labels, np.concatenate([pos_scores, neg_scores]))
print("  ROC-AUC (proyección como score):", auc)


[CAV] shape: (1024,) device: cuda:0 dtype: torch.float16
[CAV(f32)] ||cav||_2: 1.0000005960464478
[CAV(f32)] primeras 16: [-0.04107666 -0.00366783  0.03268433 -0.01062012  0.01856995 -0.04440308
 -0.02587891 -0.02815247 -0.05526733 -0.00191212 -0.03616333 -0.05267334
 -0.02157593  0.04537964 -0.05548096  0.00855255]

[CHECK] Proyecciones sobre CAV (fp32)
  pos mean/std/min/max: 42.16107006072998 2.1319360725619023 38.36150360107422 45.966835021972656
  neg mean/std/min/max: -43.955804443359376 7.687285678368057 -64.94338989257812 -38.34946060180664
  ROC-AUC (proyección como score): 1.0


In [17]:
import torch, numpy as np
from sklearn.svm import LinearSVC

CONCEPT_SPAN = " electrical component"

def to_ids(text): 
    return model.to_tokens(text, prepend_bos=False).to(next(model.parameters()).device)[0]

def find_subseq_ids(hay, nee):
    H, N = len(hay), len(nee)
    for i in range(H-N+1):
        if torch.equal(hay[i:i+N], nee):
            return list(range(i, i+N))
    return None

@torch.no_grad()
def resid_pre_on_span(prompt, span_ids, layer, hook=HOOK_NAME):
    toks = model.to_tokens(prompt, prepend_bos=False).to(next(model.parameters()).device)
    _, cache = model.run_with_cache(toks, remove_batch_dim=False,
                                    names_filter=f"blocks.{layer}.{hook}")
    H = cache[f"blocks.{layer}.{hook}"][0]     # [seq, d_model]
    idxs = find_subseq_ids(to_ids(prompt), span_ids)
    v = H[idxs, :].mean(0) if idxs else H[-1, :]  # fallback último
    return v.detach().float()  # <- SIEMPRE fp32

span_ids = to_ids(CONCEPT_SPAN)

# Dataset fp32 para SVM
X, y = [], []
with torch.no_grad():
    for s in positives:
        X.append(resid_pre_on_span(s, span_ids, LAYER_FOR_CAV).cpu().numpy()); y.append(1)
    for s in negatives:
        X.append(resid_pre_on_span(s, span_ids, LAYER_FOR_CAV).cpu().numpy()); y.append(0)
X = np.stack(X, 0).astype(np.float32)
y = np.array(y, dtype=np.int32)

clf = LinearSVC(C=1.0, random_state=42, dual=False, max_iter=5000)
clf.fit(X, y)
w = clf.coef_.reshape(-1)
w = w / (np.linalg.norm(w) + 1e-12)

# CAV fp32 para análisis y fp16/fp32 para hook (igual que el modelo)
cav_span_f32 = torch.tensor(w, device=next(model.parameters()).device, dtype=torch.float32)

# Normalizar signo: queremos mean_pos > mean_neg en proyección
def mean_proj(cav_vec):
    ps = [float(torch.dot(resid_pre_on_span(s, span_ids, LAYER_FOR_CAV), cav_vec.detach().float())) for s in positives]
    ns = [float(torch.dot(resid_pre_on_span(s, span_ids, LAYER_FOR_CAV), cav_vec.detach().float())) for s in negatives]
    return np.mean(ps), np.mean(ns)

m_pos, m_neg = mean_proj(cav_span_f32)
if m_pos < m_neg:
    cav_span_f32 = -cav_span_f32

# Guardar 2 versiones: análisis (fp32) y ejecución (dtype del modelo)
torch.save(cav_span_f32, "cav_electrical_component_span.fp32.pt")
cav_span_exec = cav_span_f32.to(dtype=next(model.parameters()).dtype)  # p.ej. fp16
torch.save(cav_span_exec, "cav_electrical_component_span.pt")

print("[RECALC] mean_pos:", float(m_pos), "mean_neg:", float(m_neg))
print("[RECALC] guardados: cav_electrical_component_span.fp32.pt y cav_electrical_component_span.pt")


[RECALC] mean_pos: 38.37613220214844 mean_neg: -39.30153827667236
[RECALC] guardados: cav_electrical_component_span.fp32.pt y cav_electrical_component_span.pt


In [18]:
import torch, torch.nn.functional as F
import numpy as np
from typing import List, Dict

LAYER_FOR_CAV = 12
HOOK_NAME = "hook_resid_pre"

def make_hook_on_positions(cav_vec, alpha, mode, positions: List[int]):
    # cav_vec llega con dtype del modelo; dentro calculamos en el mismo dtype
    u = cav_vec / (torch.norm(cav_vec) + 1e-12)
    pos = torch.tensor(sorted(set(positions)))
    def _hook(h, hook):
        v = u.to(h.device)
        idx = pos.to(h.device)
        h2 = h.clone()
        sl = h2[:, idx, :]
        if mode == "add":
            sl = sl - alpha * v
        elif mode == "project":
            proj = (sl @ v)[..., None] * v
            sl = sl - proj
        h2[:, idx, :] = sl
        return h2
    return _hook

@torch.no_grad()
def score_set(prompt: str, opts: List[str]) -> Dict[str, float]:
    dev = next(model.parameters()).device
    logps = {}
    for o in opts:
        p = prompt.rstrip() + (" " if not prompt.endswith(" ") else "")
        c = " " + o
        toks_all = model.to_tokens(p + c, prepend_bos=False).to(dev)
        P        = model.to_tokens(p, prepend_bos=False).shape[1]
        cont_ids = model.to_tokens(c, prepend_bos=False).to(dev)[0]
        logits   = model(toks_all)
        logp     = F.log_softmax(logits, dim=-1)[0]
        pref     = logp[P-1:-1]
        lp       = pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()
        logps[o] = lp
    vs = np.array(list(logps.values()), dtype=np.float64)
    vs = vs - vs.max()
    ps = np.exp(vs); ps = ps/ps.sum()
    return {k: float(p) for k,p in zip(logps.keys(), ps)}

@torch.no_grad()
def score_set_hook_only_on_option_tokens(prompt: str, opts: List[str], layer: int, cav_path: str,
                                         mode="project", alpha=1.0) -> Dict[str, float]:
    dev  = next(model.parameters()).device
    cavv = torch.load(cav_path, map_location=dev)  # dtype del modelo (fp16 si aplica)
    logps = {}
    for o in opts:
        p = prompt.rstrip() + (" " if not prompt.endswith(" ") else "")
        c = " " + o
        toks_all = model.to_tokens(p + c, prepend_bos=False).to(dev)
        P        = model.to_tokens(p, prepend_bos=False).shape[1]
        cont_ids = model.to_tokens(c, prepend_bos=False).to(dev)[0]
        opt_positions = list(range(P, P + len(cont_ids)))
        hook = make_hook_on_positions(cavv, alpha, mode, opt_positions)
        logits = model.run_with_hooks(toks_all, fwd_hooks=[(f"blocks.{layer}.{HOOK_NAME}", hook)], return_type="logits")
        logp   = F.log_softmax(logits, dim=-1)[0]
        pref   = logp[P-1:-1]
        lp     = pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item()
        logps[o] = lp
    vs = np.array(list(logps.values()), dtype=np.float64)
    vs = vs - vs.max()
    ps = np.exp(vs); ps = ps/ps.sum()
    return {k: float(p) for k,p in zip(logps.keys(), ps)}

# ——— Re-test ———
print("\n[BASELINE]")
base = score_set(PROMPT_TEST, CANDIDATES)
for k,v in base.items(): print(f"  P({k}) = {v:.4f}")

print("\n[INTERVENCIÓN: span CAV, L12, project, α=1.0, solo continuación]")
interv = score_set_hook_only_on_option_tokens(
    PROMPT_TEST, CANDIDATES, LAYER_FOR_CAV,
    cav_path="cav_electrical_component_span.pt",
    mode="project", alpha=1.0
)
for k,v in interv.items(): print(f"  P({k}) = {v:.4f}")

print("\n[Δ]")
for k in CANDIDATES:
    print(f"  Δ {k:>20s} = {interv[k]-base[k]:+.4f}")



[BASELINE]
  P(electrical component) = 0.7820
  P(animal) = 0.0710
  P(vehicle) = 0.1209
  P(furniture) = 0.0261

[INTERVENCIÓN: span CAV, L12, project, α=1.0, solo continuación]
  P(electrical component) = 0.6451
  P(animal) = 0.1157
  P(vehicle) = 0.1967
  P(furniture) = 0.0425

[Δ]
  Δ electrical component = -0.1369
  Δ               animal = +0.0446
  Δ              vehicle = +0.0759
  Δ            furniture = +0.0164


# Experimento 2

In [19]:
def make_steer_hook(cav: torch.Tensor, alpha: float, mode: str):
    # u: dirección unitaria del CAV (mismo device/dtype del modelo)
    u = cav / (torch.norm(cav) + 1e-12)

    def _hook(h, hook):
        # h: [batch, seq, d_model]
        if mode == "add":
            # RESTA DIRECCIONAL (no depende de h)
            return h - alpha * u
        elif mode == "project":
            # PROYECCIÓN ESCALADA POR α  (¡parche!: antes no escalaba)
            proj = torch.einsum("bsd,d->bs", h, u).unsqueeze(-1) * u
            return h - alpha * proj
        else:
            return h
    return _hook


In [20]:
def make_hook_on_positions(cav_vec, alpha, mode, positions):
    u = cav_vec / (torch.norm(cav_vec) + 1e-12)
    pos = torch.tensor(sorted(set(positions)))
    def _hook(h, hook):
        v = u.to(h.device)
        idx = pos.to(h.device)
        h2 = h.clone()
        sl = h2[:, idx, :]
        if mode == "add":
            sl = sl - alpha * v
        elif mode == "project":
            proj = (sl @ v)[..., None] * v
            sl = sl - alpha * proj   # <— parche
        h2[:, idx, :] = sl
        return h2
    return _hook


In [21]:
import numpy as np, pandas as pd, torch, torch.nn.functional as F

ALPHAS = [0.25, 0.5, 1.0, 2.0, 3.5, 6.0]
MODES  = ["project", "add"]
LAYER  = 12
HOOK   = "hook_resid_pre"
CAV_PATH = "cav_electrical_component_span.pt"  # el CAV “span” que guardaste

def score_once(prompt, option):
    dev = next(model.parameters()).device
    p = prompt.rstrip() + (" " if not prompt.endswith(" ") else "")
    c = " " + option
    toks_all = model.to_tokens(p + c, prepend_bos=False).to(dev)
    P        = model.to_tokens(p, prepend_bos=False).shape[1]
    cont_ids = model.to_tokens(c, prepend_bos=False).to(dev)[0]
    logits   = model(toks_all)
    logp     = F.log_softmax(logits, dim=-1)[0]
    pref     = logp[P-1:-1]
    return float(pref[torch.arange(cont_ids.shape[0]), cont_ids].sum().item())

@torch.no_grad()
def softmax_local(d: dict):
    vs = np.array(list(d.values()), dtype=np.float64)
    vs = vs - vs.max()
    ps = np.exp(vs); ps = ps / ps.sum()
    return {k: float(p) for k,p in zip(d.keys(), ps)}

@torch.no_grad()
def score_set_baseline(prompt, candidates):
    lp = {opt: score_once(prompt, opt) for opt in candidates}
    return softmax_local(lp)

@torch.no_grad()
def score_set_with_hook(prompt, candidates, layer, cav_path, mode, alpha):
    dev  = next(model.parameters()).device
    cavv = torch.load(cav_path, map_location=dev)

    res = {}
    for opt in candidates:
        p  = prompt.rstrip() + (" " if not prompt.endswith(" ") else "")
        c  = " " + opt
        ta = model.to_tokens(p + c, prepend_bos=False).to(dev)
        P  = model.to_tokens(p, prepend_bos=False).shape[1]
        ci = model.to_tokens(c, prepend_bos=False).to(dev)[0]
        positions = list(range(P, P + len(ci)))
        hook = make_hook_on_positions(cavv, alpha, mode, positions)
        logits = model.run_with_hooks(ta, fwd_hooks=[(f"blocks.{layer}.{HOOK}", hook)], return_type="logits")
        logp   = F.log_softmax(logits, dim=-1)[0]
        pref   = logp[P-1:-1]
        lp     = float(pref[torch.arange(len(ci)), ci].sum().item())
        res[opt] = lp
    return softmax_local(res)

# —— ejecutar —— 
prompt = "A fuse is an"
cands  = ["electrical component", "animal", "vehicle", "furniture"]

base = score_set_baseline(prompt, cands)
rows = []
for mode in MODES:
    for a in ALPHAS:
        hyp = score_set_with_hook(prompt, cands, LAYER, CAV_PATH, mode, a)
        rows.append({
            "mode": mode, "alpha": a,
            "P_base(electric)": base["electrical component"],
            "P_hook(electric)": hyp["electrical component"],
            "Δ_electric": hyp["electrical component"] - base["electrical component"],
        })

df_sweep = pd.DataFrame(rows).sort_values(["mode","alpha"])
print(df_sweep.to_string(index=False))
df_sweep.to_csv("dose_response_single_item.csv", index=False)


   mode  alpha  P_base(electric)  P_hook(electric)  Δ_electric
    add   0.25          0.781957          0.780622   -0.001335
    add   0.50          0.781957          0.781957    0.000000
    add   1.00          0.781957          0.780622   -0.001335
    add   2.00          0.781957          0.777934   -0.004023
    add   3.50          0.781957          0.773859   -0.008098
    add   6.00          0.781957          0.768344   -0.013613
project   0.25          0.781957          0.755593   -0.026364
project   0.50          0.781957          0.725601   -0.056355
project   1.00          0.781957          0.645066   -0.136891
project   2.00          0.781957          0.294977   -0.486979
project   3.50          0.781957          0.034742   -0.747215
project   6.00          0.781957          0.003234   -0.778723
